# Agent 2 — Notebook 05

## Final Documented Phase 1 + Phase 2 + Block-Aware Phase 3

```text
Original implementation — official-topic retrieval and MiniLM ranking
Phase 1                — original visual-question page rendering
Phase 2                — adaptive relevance, quality and duplicate gates
Phase 3 initial        — line-by-line structured mark-scheme cleanup
Phase 3 final          — wrapped-line and block-aware classification
```

All earlier stages remain documented so the complete sequence of testing, issues,
decisions and refinements can be reviewed before Streamlit integration.


## Retrieval design

This notebook uses a **metadata-first hybrid retrieval** approach:

```text
Agent 1 official topics
            ↓
exact official-reference filter in Qdrant
            ↓
MiniLM ranking within the filtered topic pool
            ↓
controlled same-section fallback when needed
            ↓
duplicate removal
            ↓
deterministic reranking
            ↓
question-count and total-marks balancing
            ↓
PostgreSQL full question + mark scheme
            ↓
final assessment package
```

The official reference determines the topic pool. MiniLM does not guess the
syllabus topic; it only ranks questions within that pool according to the
lesson-specific concept.

Example:

```text
Official reference: 3.2.2 Programming concepts
Detected concept: Iteration
```

The filter retrieves only `3.2.2` questions, and MiniLM ranks
iteration-related questions above other programming-concept questions.


## Development record — issue discovered after the first retrieval test

The original Notebook 05 pipeline successfully:

```text
validated Agent 1 topics
filtered Qdrant by official AQA references
ranked candidates with MiniLM
balanced the requested marks
fetched complete PostgreSQL question and mark-scheme records
generated JSON, CSV, Markdown and TXT outputs
```

The original logic is intentionally retained in this notebook so the development
history remains visible.

### Issue observed during output evaluation

Some retrieved questions referred to:

```text
figures
flowcharts
trace tables
diagrams
tables
```

The text output correctly stated that a visual existed, but the original image was
not attached. For example, a question could say `Figure 4 shows...` while the
student could not see Figure 4.

### Impact

```text
The question may be technically relevant but not independently solvable.
The student cannot complete a trace table or interpret a diagram without the source visual.
Manual evaluation of the retrieved assessment becomes less reliable.
```

### Options considered

```text
Option A — exclude visual questions
Option B — attach the original question-page image
Option C — use only visuals completely reconstructed from extracted text
```

### Decision

**Option B was selected.**

For every selected question marked as visual, this phase renders the relevant
original Question Paper PDF page as a PNG image and attaches its path to the
assessment package.

### Why full-page rendering is used in Phase 1

Full-page rendering is more reliable than automatic cropping because it avoids
accidentally cutting off:

```text
figure labels
table headings
continuation text
question instructions
page-spanning visual context
```

### Current scope

```text
render selected visual-question pages only
use the original Question Paper PDF
save PNG images under Agent2/OUTPUT
add image paths to JSON, Markdown, TXT and CSV outputs
display rendered images inside the notebook for evaluation
do not regenerate or modify MiniLM vectors
do not change the current retrieval/ranking logic
```

### Planned future refinement

A later phase may detect the exact question bounding box and crop the page around
the question and visual. That refinement is deliberately not hidden inside this
phase, so the supervisor can see the progression from the identified issue to the
first reliable solution.


## Phase 2 development record — concept relevance and question-quality filtering

Phase 1 solved the missing-visual problem. Manual evaluation of the resulting
assessment then identified a second set of issues.

### Baseline behaviour retained

The existing pipeline already:

```text
filters by official AQA reference
ranks with MiniLM
removes duplicates
balances total marks
fetches full PostgreSQL mark schemes
```

This baseline logic remains in the notebook as evidence of the original approach.

### Issues observed after Phase 1

```text
1. An official AQA subsection may still contain several different concepts.
2. A weak concept match may be selected because it helps reach the target marks.
3. Incomplete/parser-noisy question text may enter the final assessment.
4. "minimum primary questions" does not guarantee supporting-topic coverage.
```

Examples observed during evaluation included:

```text
a structured-programming question for a tracing-focused lesson
question text ending in "Do not"
parser fragments such as "outsid" and "bo"
five primary questions despite approved supporting topics
```

### Options considered

```text
Option A — increase the MiniLM score only
Option B — use an LLM to approve every retrieved question
Option C — deterministic hybrid gate:
           enriched lesson evidence
           + semantic threshold
           + text-quality rules
           + explicit topic-distribution constraints
```

### Decision

**Option C was selected.**

It is deterministic, auditable and suitable for later controlled evaluation.

### Phase 2 implementation

```text
1. Use actual Agent 1 source chunk text when available.
2. Use lesson summary only as a clearly recorded fallback.
3. Apply a provisional semantic relevance threshold.
4. Reject incomplete or parser-noisy question text.
5. Apply the gate before marks balancing.
6. Require supporting-topic and distinct-reference coverage.
7. Export every gate decision and rejection reason.
```

### Important limitation

The semantic threshold is provisional. Notebook 06 must compare thresholds and
human relevance ratings before any production promotion.


## Phase 2 threshold-development record — fixed to adaptive

### Initial threshold approach

The first Phase 2 implementation used two fixed MiniLM thresholds:

```text
strict threshold  = 0.60
relaxed threshold = 0.55
```

The logic was:

```text
try 0.60
    ↓ insufficient candidate pool
try 0.55
    ↓ insufficient candidate pool
use a documented quality-safe rescue or stop
```

### Issue found during testing

This approach assumed that all official topics would produce similar MiniLM score
distributions. The test showed that this assumption was not reliable.

For example:

```text
tracing questions may naturally score around 0.60–0.68
array questions may have a lower but still meaningful score range
iteration questions may produce a different score distribution again
```

A fixed `0.60` threshold could therefore retain several tracing questions while
rejecting the strongest available array questions. Lowering the global threshold to
`0.55` affected every topic, including topics that already had strong candidates.

### Options considered

```text
Option A — keep lowering one global threshold
Option B — configure a manually selected threshold for each syllabus topic
Option C — calculate a threshold from each topic's own candidate-score distribution
```

### Final decision

**Option C was adopted: per-topic adaptive thresholding.**

The fixed `0.60` and `0.55` values remain in this notebook only as part of the
documented development history. They are no longer the active Phase 2 gate.

### Adaptive threshold logic

For each approved Agent 1 topic, the notebook calculates:

```text
candidate count
quality-safe candidate count
minimum score
maximum score
median score
selected percentile score
best score minus configured margin
required candidate quota for that topic
final adaptive threshold
```

The initial topic threshold is based on:

```text
max(
    selected score percentile,
    best topic score - score margin
)
```

It is then constrained by:

```text
absolute minimum floor
maximum allowed threshold
minimum candidate quota required for topic coverage
```

### Safety rules

```text
the text-quality gate remains non-adaptive
near-duplicate removal remains non-adaptive
the adaptive threshold cannot fall below the absolute floor
a lower quality-safe rescue remains separately labelled
any selected rescue candidate forces human review
```

### Why this is less overfitted

The threshold is not hard-coded for tracing, arrays or iteration. It is derived from
the current approved topic's own retrieved candidates and the current assessment
requirements.


## Final Phase 2 refinement and Phase 3 development record

### Phase 2 issue still found after adaptive thresholding

Adaptive thresholding solved the insufficient candidate-pool problem and successfully
covered all approved topics. However, the evaluation still selected a question
containing:

```text
Complete the decomposition ... boxes and.
Figure 7
```

The previous detector inspected only the final non-empty line. Because `Figure 7`
appeared after the incomplete sentence, the problem was not detected.

### Final Phase 2 decision

The completeness detector now:

```text
normalises the complete question text
splits it into sentence-like segments
ignores standalone visual labels and syllabus headings
checks every meaningful instructional segment
flags a segment ending in a dangling connector such as "and." or "or."
keeps the rule general rather than matching one observed phrase
```

The release state is also refined:

```text
ready_for_release
    only after actual Agent 1 chunk evidence has been used

evaluation_ready
    retrieval is technically valid, but the current notebook used lesson-summary fallback
    or Phase 3 structured fields still require review

needs_user_decision
    marks are outside tolerance or a semantic rescue was selected
```

### Phase 3 issue

The complete raw `marking_guidance` was readable, but the older structured fields
sometimes mixed:

```text
worked code examples into marking points
example commentary into acceptable answers
valid code into rejected answers
examiner notes into the wrong category
```

### Phase 3 decision

The raw marking guidance remains the source of truth. Phase 3 adds a new deterministic
structured view without overwriting the original database values.

It separates:

```text
marking points
acceptable answers
rejected answers
additional guidance
worked examples
assessment objectives
```

Every selected mark scheme receives:

```text
cleanup status
rule confidence
review reasons
legacy fields for comparison
cleaned Phase 3 fields
```

Low-confidence cleanup is marked for human review rather than silently promoted.


## Phase 2 refinement record — issues found after the first Phase 2 run

The initial Phase 2 implementation successfully reduced the candidate pool and
introduced supporting-topic coverage. The first evaluated output showed:

```text
60 baseline unique candidates
10 Phase 2 eligible candidates
50 candidates rejected
strict semantic threshold 0.60 used without relaxation
3 primary and 2 supporting questions selected
```

This confirmed that the gate was active. However, the evaluation also revealed
four remaining problems.

### Issue 1 — near-duplicate questions

The same trace-table question appeared under two PMT topical packs. The records had
different IDs and a small extra topical heading, so exact hash/text deduplication did
not recognise them as duplicates.

### Issue 2 — generic incomplete question endings

One selected question ended with:

```text
boxes and.
```

The original quality gate detected known fragments such as `Do not`, `outsid` and
`bo`, but did not yet detect a generic sentence ending in a conjunction or
preposition.

### Issue 3 — not every approved topic was represented

Agent 1 approved three references:

```text
3.1.1 — Representing algorithms
3.2.6 — Data structures / arrays
3.2.2 — Programming concepts / iteration
```

The first Phase 2 run covered only two references because the request required a
minimum of two distinct references.

### Issue 4 — requested marks were not satisfied

The request asked for 20 marks, but the strongest set satisfying the quality and
topic constraints totalled 13 marks. This was not a retrieval crash, but the output
needed an explicit release status rather than silently appearing fully satisfied.

### Options considered

```text
Near duplicates:
A — rely only on content hashes
B — lexical comparison only
C — transparent hybrid comparison using cleaned text, token overlap and MiniLM

Topic coverage:
A — keep only a distinct-reference count
B — require every approved Agent 1 reference when the assessment size permits

Marks:
A — lower quality requirements automatically until the target is reached
B — return the best safe assessment and clearly require a user decision
```

### Decisions

```text
Near duplicates → Option C
Topic coverage  → cover every approved reference by default
Marks handling  → return best safe set and flag "needs_user_decision"
```

### Refinement added in this notebook

```text
1. Remove PMT headings and common formatting noise before duplicate comparison.
2. Detect exact cleaned-text matches.
3. Detect near duplicates using:
   - lexical sequence similarity
   - token Jaccard overlap
   - MiniLM question-to-question similarity
   - same-mark safeguard
4. Detect generic incomplete endings such as "and.", "or.", "to." and "for.".
5. Require all approved official references when the question count permits.
6. Separate technical notebook completion from assessment release readiness.
7. Export duplicate and release-readiness evidence for audit.
```

### Remaining limitation

The current test still uses lesson-summary fallback because actual Agent 1 chunk text
was not supplied. The retrieval code supports real chunk text, but this must be tested
during Streamlit integration.


### Final generalisation review

The first refinement used the observed `boxes and.` output as an example. The final
implementation does **not** keep a dedicated production rule for that phrase.

Instead, it applies a general sentence-completeness rule only when:

```text
the final non-empty line is short
the line does not end with a question mark
the final meaningful token is a clearly incomplete connector:
and / or / the / a / an
```

This reduces overfitting and lowers the risk of rejecting valid questions that
naturally contain words such as `for`, `with`, `by`, `to` or `of`.


## 1. Install dependencies


In [ ]:
%pip install -q "sentence-transformers>=3.0,<6" "qdrant-client>=1.12,<2" "sqlalchemy>=2.0" "psycopg[binary]>=3.1" "pymupdf>=1.24,<2" pandas numpy torch python-dotenv


## 2. Configuration and Agent 1 sample input

Replace `AGENT1_TOPIC_OUTPUT` with the actual list returned by Agent 1.

The sample below follows the same structure shown in the Agent 1 interface.


In [ ]:
from __future__ import annotations

import json
import math
import os
import re
import time
import uuid

import fitz
from datetime import datetime, timezone
from difflib import SequenceMatcher
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv
from IPython.display import Image as IPythonImage, display
from qdrant_client import QdrantClient, models
from sentence_transformers import SentenceTransformer
from sqlalchemy import (
    MetaData,
    Table,
    create_engine,
    select,
    text,
)
from sqlalchemy.engine import Engine
from sqlalchemy.orm import Session


cwd = Path.cwd().resolve()

PROJECT_ROOT = (
    cwd.parent
    if cwd.name.lower() in {"notebooks", "notebook"}
    else cwd
)

OUTPUT_DIR = PROJECT_ROOT / "OUTPUT"
MODEL_CACHE_DIR = PROJECT_ROOT / "cache" / "models"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)

load_dotenv(PROJECT_ROOT / ".env")


DATABASE_URL = os.getenv(
    "AGENT2_DATABASE_URL",
    "",
).strip()

if not DATABASE_URL:
    raise RuntimeError(
        "AGENT2_DATABASE_URL is missing from Agent2/.env"
    )


QDRANT_URL = os.getenv(
    "QDRANT_URL",
    "http://localhost:6333",
).strip()

QDRANT_API_KEY = (
    os.getenv("QDRANT_API_KEY", "").strip()
    or None
)

AGENT1_COLLECTION = os.getenv(
    "QDRANT_COLLECTION",
    "aqa_gcse_computer_science_8525",
).strip()

AGENT2_COLLECTION = (
    os.getenv(
        "AGENT2_QDRANT_COLLECTION",
        "",
    ).strip()
    or f"{AGENT1_COLLECTION}_questions"
)

QDRANT_TIMEOUT_SECONDS = int(
    os.getenv(
        "QDRANT_TIMEOUT_SECONDS",
        "30",
    )
)

raw_threshold = os.getenv(
    "QDRANT_SCORE_THRESHOLD",
    "",
).strip()

QDRANT_SCORE_THRESHOLD = (
    float(raw_threshold)
    if raw_threshold
    else None
)


MODEL_NAME = (
    "sentence-transformers/all-MiniLM-L6-v2"
)

EXPECTED_VECTOR_SIZE = 384
EXPECTED_QDRANT_POINTS = 820

SPECIFICATION_CODE = "8525"
SPECIFICATION_VERSION = (
    "first_teaching_2020_last_exams_2026"
)

RETRIEVAL_VERSION = (
    "agent2-official-topic-retrieval-v1.5.0-phase3-block-aware"
)

CANDIDATES_PER_TOPIC = 20
FALLBACK_CANDIDATES_PER_TOPIC = 10

ALLOW_SECTION_FALLBACK = True
STORE_RETRIEVAL_LOGS = True

PRIMARY_ROLE_WEIGHT = 1.00
SUPPORTING_ROLE_WEIGHT = 0.65

EXACT_REFERENCE_BONUS = 0.15
SECTION_FALLBACK_PENALTY = 0.12
HUMAN_CORRECTED_BONUS = 0.02

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


# ---------------------------------------------------------
# Phase 2 — concept relevance and question-quality gate
# ---------------------------------------------------------
ENABLE_PHASE2_GATE = True
PHASE2_VERSION = "agent2-concept-quality-gate-v1.2.0-adaptive"

# Historical values used in the initial Phase 2 test.
# They are retained for documentation only and are not used
# by the final adaptive gate.
LEGACY_FIXED_STRICT_THRESHOLD = 0.60
LEGACY_FIXED_RELAXED_THRESHOLD = 0.55
LEGACY_FIXED_THRESHOLD_APPROACH_ENABLED = False

# Final Phase 2 approach: calculate a separate threshold for
# each approved Agent 1 topic.
ENABLE_ADAPTIVE_SEMANTIC_THRESHOLD = True
ADAPTIVE_THRESHOLD_VERSION = (
    "agent2-per-topic-adaptive-threshold-v1.0.0"
)

# Topic score distribution controls
ADAPTIVE_SCORE_PERCENTILE = 60.0
ADAPTIVE_TOP_SCORE_MARGIN = 0.10

# Safety bounds
ADAPTIVE_ABSOLUTE_MINIMUM_SCORE = 0.45
ADAPTIVE_MAXIMUM_ALLOWED_THRESHOLD = 0.70

# Small numerical tolerance when matching a quota boundary.
ADAPTIVE_THRESHOLD_EPSILON = 1e-9

ENABLE_QUESTION_TEXT_QUALITY_GATE = True
MIN_QUESTION_WORD_COUNT = 8

# Streamlit can pass actual Module 2 chunk text here.
AGENT1_SOURCE_CHUNK_TEXTS: dict[int, str] = {}
MAX_QUERY_EVIDENCE_CHARACTERS = 4000

# ---------------------------------------------------------
# Final Phase 2 and Phase 3 release controls
# ---------------------------------------------------------
REQUIRE_ACTUAL_AGENT1_CHUNK_EVIDENCE_FOR_RELEASE = True

PHASE3_VERSION = (
    "agent2-mark-scheme-structured-cleanup-v1.1.0-block-aware"
)
ENABLE_PHASE3_MARK_SCHEME_CLEANUP = True
PHASE3_MIN_RULE_CONFIDENCE = 0.70
PHASE3_RAW_GUIDANCE_IS_SOURCE_OF_TRUTH = True
PHASE3_STORE_AUDIT_RECORDS = True
PHASE3_BLOCK_PARSER_VERSION = "agent2-mark-scheme-block-parser-v1.0.0"
PHASE3_MERGE_WRAPPED_LINES = True
PHASE3_SPLIT_INLINE_MARKERS = True
PHASE3_REVIEW_ON_AMBIGUOUS_BLOCKS = True


# ---------------------------------------------------------
# Phase 2 refinement — duplicate, coverage and release rules
# ---------------------------------------------------------
PHASE2_REFINEMENT_VERSION = (
    "agent2-concept-quality-gate-v1.1.0"
)

ENABLE_NEAR_DUPLICATE_GATE = True

# A duplicate must normally have the same mark allocation.
REQUIRE_SAME_MARKS_FOR_NEAR_DUPLICATES = True

NEAR_DUPLICATE_LEXICAL_THRESHOLD = 0.92
NEAR_DUPLICATE_TOKEN_JACCARD_THRESHOLD = 0.80
NEAR_DUPLICATE_SEMANTIC_THRESHOLD = 0.985

# When the number of requested questions allows it, require at
# least one selected question from every approved Agent 1 reference.
COVER_ALL_APPROVED_TOPICS = True

# A set outside this tolerance is returned for evaluation but is
# not marked ready for student release without a user decision.
TARGET_MARKS_TOLERANCE = 2
ALLOW_BEST_AVAILABLE_ASSESSMENT = True

# ---------------------------------------------------------
# Phase 2 resilience — quality-safe topic coverage rescue
# ---------------------------------------------------------
ENABLE_QUALITY_SAFE_TOPIC_RESCUE = True

# Rescue is only considered after both the strict and relaxed
# thresholds fail to produce a sufficient topic-balanced pool.
MIN_TOPIC_RESCUE_SEMANTIC_SCORE = 0.40

# Rescued candidates must still pass every text-quality rule.
# They are marked for human review and prevent automatic release.

# ---------------------------------------------------------
# Phase 1 — original question-page image rendering
# ---------------------------------------------------------
ENABLE_VISUAL_PAGE_RENDERING = True

# Full-page rendering is intentionally used in this first
# implementation. Automatic question-only cropping is a
# separately documented future refinement.
RENDER_FULL_QUESTION_PAGES = True

VISUAL_RENDER_DPI = 180
DISPLAY_RENDERED_IMAGES_IN_NOTEBOOK = True

# Notebook 02 stores PDF page numbers as human-readable,
# one-based page numbers.
DATABASE_PAGE_NUMBERS_ARE_ONE_BASED = True

VISUAL_RENDERING_VERSION = (
    "agent2-visual-question-pages-v1.0.0"
)


AGENT1_TOPIC_OUTPUT = [
    {
        "topic": (
            "Algorithm tracing and program execution"
        ),
        "role": "primary",
        "official_reference": "3.1.1",
        "confidence": 0.8172,
        "ranking_score": 0.6502,
        "source_chunks": [3, 4, 5, 6, 7, 8],
    },
    {
        "topic": (
            "One- and two-dimensional arrays"
        ),
        "role": "supporting",
        "official_reference": "3.2.6",
        "confidence": 0.8477,
        "ranking_score": 0.6109,
        "source_chunks": [2, 3, 5, 6, 10],
    },
    {
        "topic": "Iteration",
        "role": "supporting",
        "official_reference": "3.2.2",
        "confidence": 0.5943,
        "ranking_score": 0.3768,
        "source_chunks": [3, 8],
    },
]


LESSON_SUMMARY = (
    "The lesson focused on tracing algorithms and following "
    "program execution. It also covered one- and "
    "two-dimensional arrays and iteration."
)


ASSESSMENT_REQUEST = {
    "number_of_questions": 5,
    "target_total_marks": 20,
    "minimum_question_marks": 1,
    "maximum_question_marks": 12,
    "minimum_primary_questions": 3,

    # Phase 2 topic-distribution requirements
    "minimum_supporting_questions": 1,
    "minimum_distinct_official_references": 2,

    "include_code_questions": True,
    "include_visual_questions": True,

    # Set to "1B" for Python Paper 1 only,
    # "2" for Paper 2 only, or None for both.
    "paper_code": None,

    # Set to "Python" when required,
    # otherwise keep None.
    "programming_language": None,
}


print(f"Project root:       {PROJECT_ROOT}")
print(f"Qdrant collection:  {AGENT2_COLLECTION}")
print(f"Embedding model:    {MODEL_NAME}")
print(f"Device:             {DEVICE}")
print(
    f"Visual page rendering: "
    f"{ENABLE_VISUAL_PAGE_RENDERING}"
)
print(f"Visual render DPI:   {VISUAL_RENDER_DPI}")


print(f"Phase 2 enabled:     {ENABLE_PHASE2_GATE}")
print(
    "Legacy fixed thresholds: "
    f"{LEGACY_FIXED_STRICT_THRESHOLD} / "
    f"{LEGACY_FIXED_RELAXED_THRESHOLD} "
    "(documentation only)"
)
print(
    f"Adaptive thresholding: "
    f"{ENABLE_ADAPTIVE_SEMANTIC_THRESHOLD}"
)
print(
    f"Adaptive percentile: "
    f"{ADAPTIVE_SCORE_PERCENTILE}"
)
print(
    f"Adaptive score margin: "
    f"{ADAPTIVE_TOP_SCORE_MARGIN}"
)
print(
    "Adaptive safety range: "
    f"{ADAPTIVE_ABSOLUTE_MINIMUM_SCORE}–"
    f"{ADAPTIVE_MAXIMUM_ALLOWED_THRESHOLD}"
)

print(
    f"Near-duplicate gate: "
    f"{ENABLE_NEAR_DUPLICATE_GATE}"
)
print(
    f"Cover all approved topics: "
    f"{COVER_ALL_APPROVED_TOPICS}"
)
print(
    f"Target marks tolerance: "
    f"{TARGET_MARKS_TOLERANCE}"
)
print(
    f"Quality-safe topic rescue: "
    f"{ENABLE_QUALITY_SAFE_TOPIC_RESCUE}"
)
print(
    f"Minimum rescue semantic score: "
    f"{MIN_TOPIC_RESCUE_SEMANTIC_SCORE}"
)

print(
    f"Require actual Agent 1 chunk evidence for release: "
    f"{REQUIRE_ACTUAL_AGENT1_CHUNK_EVIDENCE_FOR_RELEASE}"
)
print(f"Phase 3 cleanup enabled: {ENABLE_PHASE3_MARK_SCHEME_CLEANUP}")
print(f"Phase 3 minimum confidence: {PHASE3_MIN_RULE_CONFIDENCE}")
print(f"Phase 3 block parser: {PHASE3_BLOCK_PARSER_VERSION}")
print(f"Merge wrapped MS lines: {PHASE3_MERGE_WRAPPED_LINES}")
print(f"Split inline MS markers: {PHASE3_SPLIT_INLINE_MARKERS}")

display(pd.DataFrame(AGENT1_TOPIC_OUTPUT))
display(pd.DataFrame([ASSESSMENT_REQUEST]))


## 3. Connect to PostgreSQL and Qdrant

PostgreSQL is the source of truth for complete questions and mark schemes.
Qdrant is used for filtered semantic retrieval.


In [ ]:
engine: Engine = create_engine(
    DATABASE_URL,
    pool_pre_ping=True,
    future=True,
)

metadata = MetaData()

topics = Table(
    "assessment_topical_topics",
    metadata,
    autoload_with=engine,
)

questions = Table(
    "assessment_topical_questions",
    metadata,
    autoload_with=engine,
)

mark_schemes = Table(
    "assessment_topical_mark_scheme_entries",
    metadata,
    autoload_with=engine,
)

question_ms_links = Table(
    "assessment_topical_question_mark_scheme_links",
    metadata,
    autoload_with=engine,
)

official_mappings = Table(
    "assessment_topic_official_mappings",
    metadata,
    autoload_with=engine,
)


documents = Table(
    "assessment_topical_documents",
    metadata,
    autoload_with=engine,
)


def require_table_column(
    table: Table,
    candidates: list[str],
    purpose: str,
):
    for candidate in candidates:
        if candidate in table.c.keys():
            return table.c[candidate]

    raise RuntimeError(
        f"Could not find a column for {purpose}. "
        f"Checked {candidates}; available columns are "
        f"{list(table.c.keys())}."
    )


def optional_table_column(
    table: Table,
    candidates: list[str],
):
    for candidate in candidates:
        if candidate in table.c.keys():
            return table.c[candidate]

    return None


QUESTION_DOCUMENT_FK_COLUMN = require_table_column(
    questions,
    [
        "question_document_id",
        "document_id",
        "source_document_id",
    ],
    "the question-paper document foreign key",
)

DOCUMENT_PATH_COLUMN = require_table_column(
    documents,
    [
        "local_cache_path",
        "cache_path",
        "local_path",
        "file_path",
    ],
    "the locally cached PDF path",
)

DOCUMENT_FILE_NAME_COLUMN = optional_table_column(
    documents,
    [
        "file_name",
        "filename",
        "document_name",
    ],
)

DOCUMENT_SOURCE_URL_COLUMN = optional_table_column(
    documents,
    [
        "source_url",
        "url",
    ],
)

with engine.connect() as connection:
    connection.exec_driver_sql("SELECT 1")

print("PostgreSQL connection successful.")

print(
    "Question-document FK column: "
    f"{QUESTION_DOCUMENT_FK_COLUMN.name}"
)
print(
    "Document PDF-path column: "
    f"{DOCUMENT_PATH_COLUMN.name}"
)



qdrant_kwargs: dict[str, Any] = {
    "url": QDRANT_URL,
    "timeout": QDRANT_TIMEOUT_SECONDS,
}

if QDRANT_API_KEY is not None:
    qdrant_kwargs["api_key"] = QDRANT_API_KEY

qdrant_client = QdrantClient(
    **qdrant_kwargs
)

collection_names = {
    collection.name
    for collection
    in qdrant_client
    .get_collections()
    .collections
}

if AGENT2_COLLECTION not in collection_names:
    raise RuntimeError(
        f"Qdrant collection not found: {AGENT2_COLLECTION}"
    )

qdrant_point_count = int(
    qdrant_client.count(
        collection_name=AGENT2_COLLECTION,
        exact=True,
    ).count
)

print("Qdrant connection successful.")
print(f"Qdrant points: {qdrant_point_count}")

if qdrant_point_count != EXPECTED_QDRANT_POINTS:
    print(
        "Warning: Qdrant point count differs from "
        f"the expected {EXPECTED_QDRANT_POINTS}."
    )


## 4. Load the same MiniLM model used in Notebook 04


In [ ]:
model_started = time.perf_counter()

model = SentenceTransformer(
    MODEL_NAME,
    device=DEVICE,
    cache_folder=str(MODEL_CACHE_DIR),
)

VECTOR_SIZE = int(
    model.get_sentence_embedding_dimension()
)

if VECTOR_SIZE != EXPECTED_VECTOR_SIZE:
    raise RuntimeError(
        f"Expected {EXPECTED_VECTOR_SIZE} dimensions, "
        f"but received {VECTOR_SIZE}."
    )

print(
    f"MiniLM loaded in "
    f"{time.perf_counter() - model_started:.2f}s"
)
print(f"Vector size: {VECTOR_SIZE}")


## 5. Validate Agent 1 topics and assessment request

The official AQA reference is the canonical connection key.

This cell verifies that every reference returned by Agent 1 exists in the
approved Notebook 04A mapping table.


In [ ]:
def safe_float(
    value: Any,
    default: float = 0.0,
) -> float:
    try:
        number = float(value)
    except (TypeError, ValueError):
        return default

    return (
        number
        if math.isfinite(number)
        else default
    )


def safe_list(value: Any) -> list[Any]:
    if value is None:
        return []

    if isinstance(value, list):
        return value

    if isinstance(value, tuple):
        return list(value)

    return [value]


def normalise_agent1_topics(
    raw_topics: list[dict[str, Any]],
) -> pd.DataFrame:
    rows = []

    for index, item in enumerate(
        raw_topics,
        start=1,
    ):
        topic_name = (
            item.get("topic")
            or item.get("topic_name")
            or item.get("name")
        )

        reference = (
            item.get("official_reference")
            or item.get("reference")
        )

        role = str(
            item.get("role", "supporting")
        ).strip().lower()

        if role not in {
            "primary",
            "supporting",
        }:
            raise ValueError(
                f"Invalid role for topic {index}: {role}"
            )

        if not topic_name or not reference:
            raise ValueError(
                f"Topic {index} is missing a name or "
                "official reference."
            )

        confidence = safe_float(
            item.get("confidence"),
            0.0,
        )

        ranking_score = safe_float(
            item.get("ranking_score"),
            confidence,
        )

        if not 0.0 <= confidence <= 1.0:
            raise ValueError(
                f"Confidence outside 0–1 for {topic_name}."
            )

        rows.append(
            {
                "agent1_topic_index": index,
                "detected_topic": str(
                    topic_name
                ).strip(),
                "role": role,
                "official_reference": str(
                    reference
                ).strip(),
                "confidence": confidence,
                "ranking_score": ranking_score,
                "source_chunks": safe_list(
                    item.get("source_chunks")
                ),
                "source_chunk_texts": safe_list(
                    item.get("source_chunk_texts")
                ),
            }
        )

    frame = pd.DataFrame(rows)

    if frame.empty:
        raise RuntimeError(
            "Agent 1 returned no topics."
        )

    if not frame["role"].eq("primary").any():
        raise RuntimeError(
            "At least one primary topic is required."
        )

    return frame


agent1_topics_df = normalise_agent1_topics(
    AGENT1_TOPIC_OUTPUT
)


mapping_query = (
    select(
        official_mappings.c.topic_id,
        official_mappings.c.official_reference,
        official_mappings.c.official_concept_name,
        official_mappings.c.official_section_reference,
        official_mappings.c.official_section_name,
        official_mappings.c.pmt_subtopic_code,
        official_mappings.c.pmt_subtopic_name,
    )
    .where(
        official_mappings.c.specification_code
        == SPECIFICATION_CODE,
        official_mappings.c.specification_version
        == SPECIFICATION_VERSION,
        official_mappings.c.mapping_status
        == "approved",
        official_mappings.c.human_approved
        .is_(True),
    )
)

with engine.connect() as connection:
    mapping_df = pd.read_sql(
        mapping_query,
        connection,
    )

mapping_df["topic_id"] = (
    mapping_df["topic_id"].astype(str)
)


validated_topics_df = (
    agent1_topics_df.merge(
        mapping_df,
        on="official_reference",
        how="left",
        validate="many_to_one",
    )
)

unknown_topics_df = validated_topics_df[
    validated_topics_df["topic_id"].isna()
]

if not unknown_topics_df.empty:
    display(unknown_topics_df)

    raise RuntimeError(
        "Agent 1 returned unknown official references."
    )


validated_topics_df["role_weight"] = (
    validated_topics_df["role"].map(
        {
            "primary": PRIMARY_ROLE_WEIGHT,
            "supporting": SUPPORTING_ROLE_WEIGHT,
        }
    )
)


request = dict(ASSESSMENT_REQUEST)

integer_fields = [
    "number_of_questions",
    "target_total_marks",
    "minimum_question_marks",
    "maximum_question_marks",
    "minimum_primary_questions",
    "minimum_supporting_questions",
    "minimum_distinct_official_references",
]

for field in integer_fields:
    request[field] = int(request[field])

if request["number_of_questions"] <= 0:
    raise ValueError(
        "number_of_questions must be positive."
    )

if (
    request["minimum_primary_questions"]
    > request["number_of_questions"]
):
    raise ValueError(
        "minimum_primary_questions cannot exceed "
        "number_of_questions."
    )

if (
    request["minimum_supporting_questions"]
    > request["number_of_questions"]
):
    raise ValueError(
        "minimum_supporting_questions cannot exceed "
        "number_of_questions."
    )

if (
    request["minimum_primary_questions"]
    + request["minimum_supporting_questions"]
    > request["number_of_questions"]
):
    raise ValueError(
        "The primary and supporting minimums exceed "
        "the requested question count."
    )

if (
    request["minimum_distinct_official_references"]
    > validated_topics_df["official_reference"].nunique()
):
    raise ValueError(
        "The requested distinct-reference count exceeds "
        "the number of approved Agent 1 references."
    )

if (
    request["minimum_supporting_questions"] > 0
    and not validated_topics_df["role"].eq("supporting").any()
):
    raise ValueError(
        "Supporting-topic coverage was requested but no "
        "supporting topic was approved."
    )


approved_official_references = sorted(
    validated_topics_df[
        "official_reference"
    ]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

if (
    COVER_ALL_APPROVED_TOPICS
    and request["number_of_questions"]
    >= len(approved_official_references)
):
    request[
        "minimum_distinct_official_references"
    ] = len(
        approved_official_references
    )

    request[
        "required_official_references"
    ] = approved_official_references

    request[
        "topic_coverage_mode"
    ] = "all_approved_references"

else:
    request[
        "required_official_references"
    ] = []

    request[
        "topic_coverage_mode"
    ] = "minimum_distinct_references"


print(
    "Effective topic coverage mode: "
    f"{request['topic_coverage_mode']}"
)

print(
    "Required official references: "
    f"{request['required_official_references']}"
)

print(
    "Effective minimum distinct references: "
    f"{request['minimum_distinct_official_references']}"
)


display(
    validated_topics_df[
        [
            "detected_topic",
            "role",
            "official_reference",
            "official_concept_name",
            "official_section_reference",
            "confidence",
            "ranking_score",
        ]
    ]
)


## 6. Build MiniLM query vectors

One query is built for each Agent 1 topic. The query includes the official
section, official subsection, detected concept and lesson summary.


In [ ]:

def clean_evidence_text(value: Any) -> str:
    return re.sub(
        r"\s+",
        " ",
        str(value or ""),
    ).strip()


def collect_topic_evidence(
    row: pd.Series,
) -> tuple[str, str, int]:
    direct_texts = [
        clean_evidence_text(value)
        for value in (
            row.get("source_chunk_texts")
            if isinstance(
                row.get("source_chunk_texts"),
                list,
            )
            else []
        )
        if clean_evidence_text(value)
    ]

    mapped_texts = []

    for chunk_id in (
        row.get("source_chunks")
        if isinstance(row.get("source_chunks"), list)
        else []
    ):
        try:
            key = int(chunk_id)
        except (TypeError, ValueError):
            continue

        text_value = clean_evidence_text(
            AGENT1_SOURCE_CHUNK_TEXTS.get(key)
        )

        if text_value:
            mapped_texts.append(text_value)

    if direct_texts:
        evidence_parts = direct_texts
        evidence_source = "agent1_topic_source_chunk_texts"
    elif mapped_texts:
        evidence_parts = mapped_texts
        evidence_source = "agent1_source_chunk_text_map"
    else:
        evidence_parts = [
            clean_evidence_text(LESSON_SUMMARY)
        ]
        evidence_source = "lesson_summary_fallback"

    evidence = "\n".join(
        value
        for value in evidence_parts
        if value
    )[:MAX_QUERY_EVIDENCE_CHARACTERS]

    return evidence, evidence_source, len(evidence_parts)


evidence_results = validated_topics_df.apply(
    collect_topic_evidence,
    axis=1,
)

validated_topics_df["query_evidence"] = [
    value[0]
    for value in evidence_results
]
validated_topics_df["query_evidence_source"] = [
    value[1]
    for value in evidence_results
]
validated_topics_df["query_evidence_item_count"] = [
    value[2]
    for value in evidence_results
]


def build_query_text(row: pd.Series) -> str:
    return "\n".join(
        [
            "AQA GCSE Computer Science assessment question.",
            (
                f"Official section: {row['official_section_name']} "
                f"({row['official_section_reference']})."
            ),
            (
                f"Official topic: {row['official_concept_name']} "
                f"({row['official_reference']})."
            ),
            f"Detected lesson concept: {row['detected_topic']}.",
            f"Agent 1 topic role: {row['role']}.",
            "Lesson evidence:",
            str(row["query_evidence"]),
        ]
    )


validated_topics_df["query_text"] = (
    validated_topics_df.apply(
        build_query_text,
        axis=1,
    )
)

query_vectors = model.encode(
    validated_topics_df["query_text"].tolist(),
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False,
).astype(np.float32)

if query_vectors.shape != (
    len(validated_topics_df),
    VECTOR_SIZE,
):
    raise RuntimeError(
        "Unexpected query-vector shape."
    )

phase2_query_evidence_df = validated_topics_df[
    [
        "detected_topic",
        "role",
        "official_reference",
        "source_chunks",
        "query_evidence_source",
        "query_evidence_item_count",
        "query_evidence",
        "query_text",
    ]
].copy()

display(phase2_query_evidence_df)

fallback_evidence_topic_count = int(
    (
        validated_topics_df["query_evidence_source"]
        == "lesson_summary_fallback"
    ).sum()
)

print(
    "Topics using actual chunk evidence: "
    f"{len(validated_topics_df) - fallback_evidence_topic_count}"
)
print(
    "Topics using lesson-summary fallback: "
    f"{fallback_evidence_topic_count}"
)


## 7. Retrieve exact-reference candidates

Qdrant is filtered first by:

```text
specification code
specification version
official reference
requested marks range
optional paper/language/code/visual filters
```


In [ ]:
def exact_match(
    key: str,
    value: Any,
) -> models.FieldCondition:
    return models.FieldCondition(
        key=key,
        match=models.MatchValue(
            value=value
        ),
    )


def build_filter(
    *,
    official_reference: str | None = None,
    section_reference: str | None = None,
) -> models.Filter:
    conditions = [
        exact_match(
            "official_specification_code",
            SPECIFICATION_CODE,
        ),
        exact_match(
            "official_specification_version",
            SPECIFICATION_VERSION,
        ),
        models.FieldCondition(
            key="marks",
            range=models.Range(
                gte=request[
                    "minimum_question_marks"
                ],
                lte=request[
                    "maximum_question_marks"
                ],
            ),
        ),
    ]

    if official_reference is not None:
        conditions.append(
            exact_match(
                "official_reference",
                official_reference,
            )
        )

    if section_reference is not None:
        conditions.append(
            exact_match(
                "official_section_reference",
                section_reference,
            )
        )

    if request.get("paper_code"):
        conditions.append(
            exact_match(
                "paper_code",
                request["paper_code"],
            )
        )

    if request.get("programming_language"):
        conditions.append(
            exact_match(
                "programming_language",
                request[
                    "programming_language"
                ],
            )
        )

    if not request[
        "include_code_questions"
    ]:
        conditions.append(
            exact_match(
                "has_code",
                False,
            )
        )

    if not request[
        "include_visual_questions"
    ]:
        conditions.append(
            exact_match(
                "has_visual",
                False,
            )
        )

    return models.Filter(
        must=conditions
    )


def search_points(
    vector: np.ndarray,
    query_filter: models.Filter,
    limit: int,
) -> list[Any]:
    result = qdrant_client.query_points(
        collection_name=AGENT2_COLLECTION,
        query=vector.tolist(),
        query_filter=query_filter,
        limit=limit,
        score_threshold=QDRANT_SCORE_THRESHOLD,
        with_payload=True,
        with_vectors=False,
    )

    return (
        result.points
        if hasattr(result, "points")
        else list(result)
    )


def point_to_candidate(
    *,
    point: Any,
    topic_row: pd.Series,
    stage: str,
) -> dict[str, Any]:
    payload = point.payload or {}

    return {
        "question_id": str(
            payload.get(
                "question_id",
                point.id,
            )
        ),
        "agent1_topic_index": int(
            topic_row["agent1_topic_index"]
        ),
        "detected_topic": (
            topic_row["detected_topic"]
        ),
        "agent1_role": topic_row["role"],
        "agent1_confidence": float(
            topic_row["confidence"]
        ),
        "agent1_ranking_score": float(
            topic_row["ranking_score"]
        ),
        "role_weight": float(
            topic_row["role_weight"]
        ),
        "source_chunks": (
            topic_row["source_chunks"]
        ),
        "query_evidence_source": (
            topic_row["query_evidence_source"]
        ),
        "query_evidence": (
            topic_row["query_evidence"]
        ),
        "requested_official_reference": (
            topic_row[
                "official_reference"
            ]
        ),
        "requested_section_reference": (
            topic_row[
                "official_section_reference"
            ]
        ),
        "retrieval_stage": stage,
        "semantic_score": float(
            point.score
        ),

        "question_content_hash": (
            payload.get(
                "question_content_hash"
            )
        ),
        "embedding_text_hash": (
            payload.get(
                "embedding_text_hash"
            )
        ),
        "official_reference": (
            payload.get(
                "official_reference"
            )
        ),
        "official_concept_name": (
            payload.get(
                "official_concept_name"
            )
        ),
        "official_section_reference": (
            payload.get(
                "official_section_reference"
            )
        ),
        "question_number": (
            payload.get("question_number")
        ),
        "question_text": (
            payload.get("question_text")
        ),
        "marks": int(
            payload.get("marks", 0)
        ),
        "has_code": bool(
            payload.get("has_code", False)
        ),
        "has_visual": bool(
            payload.get(
                "has_visual",
                False,
            )
        ),
        "paper_code": (
            payload.get("paper_code")
        ),
        "programming_language": (
            payload.get(
                "programming_language"
            )
        ),
        "review_status": (
            payload.get("review_status")
        ),
    }


candidate_rows = []

for position, topic_row in (
    validated_topics_df
    .reset_index(drop=True)
    .iterrows()
):
    exact_points = search_points(
        query_vectors[position],
        build_filter(
            official_reference=(
                topic_row[
                    "official_reference"
                ]
            )
        ),
        CANDIDATES_PER_TOPIC,
    )

    for point in exact_points:
        candidate_rows.append(
            point_to_candidate(
                point=point,
                topic_row=topic_row,
                stage=(
                    "exact_official_reference"
                ),
            )
        )


exact_candidates_df = pd.DataFrame(
    candidate_rows
)

if exact_candidates_df.empty:
    raise RuntimeError(
        "No exact-reference candidates were found."
    )

print(
    f"Exact candidates: "
    f"{len(exact_candidates_df)}"
)

display(
    exact_candidates_df[
        [
            "detected_topic",
            "agent1_role",
            "official_reference",
            "question_number",
            "marks",
            "semantic_score",
            "question_text",
        ]
    ].head(50)
)


## 8. Controlled same-section fallback

Fallback runs only when a topic returns fewer candidates than the requested
question count. Fallback questions receive a ranking penalty.


In [ ]:
fallback_rows = []

if ALLOW_SECTION_FALLBACK:
    for position, topic_row in (
        validated_topics_df
        .reset_index(drop=True)
        .iterrows()
    ):
        exact_count = int(
            (
                exact_candidates_df[
                    "agent1_topic_index"
                ]
                == topic_row[
                    "agent1_topic_index"
                ]
            ).sum()
        )

        if (
            exact_count
            >= request["number_of_questions"]
        ):
            continue

        section_points = search_points(
            query_vectors[position],
            build_filter(
                section_reference=(
                    topic_row[
                        "official_section_reference"
                    ]
                )
            ),
            FALLBACK_CANDIDATES_PER_TOPIC,
        )

        for point in section_points:
            payload = point.payload or {}

            if (
                payload.get(
                    "official_reference"
                )
                == topic_row[
                    "official_reference"
                ]
            ):
                continue

            fallback_rows.append(
                point_to_candidate(
                    point=point,
                    topic_row=topic_row,
                    stage=(
                        "same_section_fallback"
                    ),
                )
            )


fallback_candidates_df = pd.DataFrame(
    fallback_rows
)

all_candidates_df = pd.concat(
    [
        exact_candidates_df,
        fallback_candidates_df,
    ],
    ignore_index=True,
)

print(
    f"Fallback candidates: "
    f"{len(fallback_candidates_df)}"
)
print(
    f"Total raw candidates: "
    f"{len(all_candidates_df)}"
)


## 9. Deterministic reranking and duplicate removal

The final score combines semantic similarity, Agent 1 confidence, Agent 1
ranking score, role priority and retrieval stage.

Duplicate questions are removed using content hash, embedding hash, or
normalised question text and marks.


In [ ]:
def final_score(
    row: pd.Series,
) -> float:
    exact_bonus = (
        EXACT_REFERENCE_BONUS
        if row["retrieval_stage"]
        == "exact_official_reference"
        else 0.0
    )

    fallback_penalty = (
        SECTION_FALLBACK_PENALTY
        if row["retrieval_stage"]
        == "same_section_fallback"
        else 0.0
    )

    corrected_bonus = (
        HUMAN_CORRECTED_BONUS
        if row["review_status"]
        == "human_corrected"
        else 0.0
    )

    score = (
        0.55 * float(
            row["semantic_score"]
        )
        + 0.15 * float(
            row["agent1_confidence"]
        )
        + 0.10 * float(
            row["agent1_ranking_score"]
        )
        + 0.20 * float(
            row["role_weight"]
        )
        + exact_bonus
        + corrected_bonus
        - fallback_penalty
    )

    return round(score, 6)


def normalise_text(value: Any) -> str:
    cleaned = re.sub(
        r"\s+",
        " ",
        str(value or "").lower(),
    ).strip()

    return re.sub(
        r"[^a-z0-9 ]+",
        "",
        cleaned,
    )


def duplicate_key(
    row: pd.Series,
) -> str:
    content_hash = str(
        row.get(
            "question_content_hash"
        )
        or ""
    ).strip()

    if content_hash:
        return f"content:{content_hash}"

    embedding_hash = str(
        row.get(
            "embedding_text_hash"
        )
        or ""
    ).strip()

    if embedding_hash:
        return f"embedding:{embedding_hash}"

    return (
        f"text:{normalise_text(row['question_text'])}:"
        f"{int(row['marks'])}"
    )


all_candidates_df["final_score"] = (
    all_candidates_df.apply(
        final_score,
        axis=1,
    )
)

all_candidates_df = (
    all_candidates_df.sort_values(
        [
            "final_score",
            "semantic_score",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)

all_candidates_df["raw_rank"] = (
    np.arange(
        1,
        len(all_candidates_df) + 1,
    )
)

all_candidates_df["duplicate_key"] = (
    all_candidates_df.apply(
        duplicate_key,
        axis=1,
    )
)

raw_count = len(all_candidates_df)

unique_candidates_df = (
    all_candidates_df.drop_duplicates(
        subset=["duplicate_key"],
        keep="first",
    )
    .reset_index(drop=True)
)

unique_candidates_df["unique_rank"] = (
    np.arange(
        1,
        len(unique_candidates_df) + 1,
    )
)

duplicates_removed = (
    raw_count
    - len(unique_candidates_df)
)

print(f"Raw candidates:      {raw_count}")
print(
    f"Duplicates removed:  "
    f"{duplicates_removed}"
)
print(
    f"Unique candidates:   "
    f"{len(unique_candidates_df)}"
)

display(
    unique_candidates_df[
        [
            "unique_rank",
            "detected_topic",
            "agent1_role",
            "retrieval_stage",
            "official_reference",
            "marks",
            "semantic_score",
            "final_score",
            "question_text",
        ]
    ].head(50)
)


## 10. Phase 2 refinement — near-duplicate detection

The original duplicate logic is retained above. It removes exact content-hash,
embedding-hash and normalised-text duplicates.

The evaluated output showed that the same question may still appear in different PMT
topical packs with:

```text
different database IDs
different topical headings
small formatting differences
the same actual question and marks
```

This refinement therefore performs a second transparent duplicate pass.

### Comparison stages

```text
1. Remove topical headings and common footer fragments.
2. Compare cleaned question text.
3. Compare token overlap.
4. Compare MiniLM question-to-question embeddings.
5. Require the same marks unless configured otherwise.
6. Keep the higher-ranked version and record the rejected duplicate.
```

Every decision is stored in a manifest rather than being silently discarded.


In [ ]:

PMT_TOPIC_HEADING_PATTERN = re.compile(
    r"^\s*\d+(?:\.\d+)+\s+[A-Za-z].*$",
    flags=re.IGNORECASE,
)

DUPLICATE_NOISE_LINES = {
    "do not",
    "turn over",
    "outside the box",
    "do not write outside the box",
}


def canonical_duplicate_text(
    value: Any,
) -> str:
    raw_text = str(
        value
        or ""
    )

    retained_lines = []

    for line in raw_text.splitlines():
        cleaned_line = re.sub(
            r"\s+",
            " ",
            line.strip(),
        )

        if not cleaned_line:
            continue

        lowered_line = cleaned_line.lower()

        if (
            PMT_TOPIC_HEADING_PATTERN.match(
                cleaned_line
            )
        ):
            continue

        if lowered_line in DUPLICATE_NOISE_LINES:
            continue

        retained_lines.append(
            cleaned_line
        )

    joined = " ".join(
        retained_lines
    ).lower()

    joined = re.sub(
        r"[^a-z0-9£≤≥←→+\-*/= ]+",
        " ",
        joined,
    )

    return re.sub(
        r"\s+",
        " ",
        joined,
    ).strip()


def token_set(
    text_value: str,
) -> set[str]:
    return set(
        re.findall(
            r"[a-z0-9]+",
            text_value.lower(),
        )
    )


def token_jaccard(
    left: str,
    right: str,
) -> float:
    left_tokens = token_set(left)
    right_tokens = token_set(right)

    if (
        not left_tokens
        and not right_tokens
    ):
        return 1.0

    union = (
        left_tokens
        | right_tokens
    )

    if not union:
        return 0.0

    return len(
        left_tokens
        & right_tokens
    ) / len(union)


def lexical_similarity(
    left: str,
    right: str,
) -> float:
    return float(
        SequenceMatcher(
            None,
            left,
            right,
        ).ratio()
    )


def detect_near_duplicates(
    candidates: pd.DataFrame,
) -> tuple[
    pd.DataFrame,
    pd.DataFrame,
]:
    working = (
        candidates.copy()
        .sort_values(
            [
                "final_score",
                "semantic_score",
            ],
            ascending=False,
        )
        .reset_index(drop=True)
    )

    working[
        "duplicate_canonical_text"
    ] = working[
        "question_text"
    ].map(
        canonical_duplicate_text
    )

    canonical_texts = working[
        "duplicate_canonical_text"
    ].tolist()

    if canonical_texts:
        duplicate_vectors = model.encode(
            canonical_texts,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        ).astype(np.float32)

    else:
        duplicate_vectors = np.empty(
            (
                0,
                VECTOR_SIZE,
            ),
            dtype=np.float32,
        )

    keep_indexes: list[int] = []
    decision_rows: list[
        dict[str, Any]
    ] = []

    for candidate_index, row in (
        working.iterrows()
    ):
        duplicate_of_index = None
        duplicate_method = None
        best_lexical = 0.0
        best_jaccard = 0.0
        best_semantic = 0.0

        for kept_index in keep_indexes:
            kept_row = working.loc[
                kept_index
            ]

            if (
                REQUIRE_SAME_MARKS_FOR_NEAR_DUPLICATES
                and int(row["marks"])
                != int(kept_row["marks"])
            ):
                continue

            left_text = row[
                "duplicate_canonical_text"
            ]
            right_text = kept_row[
                "duplicate_canonical_text"
            ]

            lexical_score = (
                lexical_similarity(
                    left_text,
                    right_text,
                )
            )

            jaccard_score = token_jaccard(
                left_text,
                right_text,
            )

            semantic_score = float(
                np.dot(
                    duplicate_vectors[
                        candidate_index
                    ],
                    duplicate_vectors[
                        kept_index
                    ],
                )
            )

            best_lexical = max(
                best_lexical,
                lexical_score,
            )
            best_jaccard = max(
                best_jaccard,
                jaccard_score,
            )
            best_semantic = max(
                best_semantic,
                semantic_score,
            )

            exact_cleaned_match = bool(
                left_text
                and left_text == right_text
            )

            strong_lexical_match = bool(
                lexical_score
                >= NEAR_DUPLICATE_LEXICAL_THRESHOLD
                and jaccard_score
                >= NEAR_DUPLICATE_TOKEN_JACCARD_THRESHOLD
            )

            strong_semantic_match = bool(
                semantic_score
                >= NEAR_DUPLICATE_SEMANTIC_THRESHOLD
                and jaccard_score
                >= 0.75
            )

            if exact_cleaned_match:
                duplicate_of_index = kept_index
                duplicate_method = (
                    "exact_cleaned_text"
                )
                break

            if strong_lexical_match:
                duplicate_of_index = kept_index
                duplicate_method = (
                    "lexical_and_token_overlap"
                )
                break

            if strong_semantic_match:
                duplicate_of_index = kept_index
                duplicate_method = (
                    "minilm_and_token_overlap"
                )
                break

        is_duplicate = (
            duplicate_of_index
            is not None
        )

        if not is_duplicate:
            keep_indexes.append(
                candidate_index
            )

        duplicate_of_question_id = (
            str(
                working.loc[
                    duplicate_of_index,
                    "question_id",
                ]
            )
            if is_duplicate
            else None
        )

        decision_rows.append(
            {
                "question_id": str(
                    row["question_id"]
                ),
                "agent1_topic_index": int(
                    row[
                        "agent1_topic_index"
                    ]
                ),
                "near_duplicate_status": (
                    "duplicate_removed"
                    if is_duplicate
                    else "retained"
                ),
                "duplicate_of_question_id": (
                    duplicate_of_question_id
                ),
                "duplicate_detection_method": (
                    duplicate_method
                ),
                "maximum_lexical_similarity": round(
                    best_lexical,
                    6,
                ),
                "maximum_token_jaccard": round(
                    best_jaccard,
                    6,
                ),
                "maximum_minilm_similarity": round(
                    best_semantic,
                    6,
                ),
                "duplicate_canonical_text": (
                    row[
                        "duplicate_canonical_text"
                    ]
                ),
            }
        )

    decisions_df = pd.DataFrame(
        decision_rows
    )

    manifest_df = working.merge(
        decisions_df,
        on=[
            "question_id",
            "agent1_topic_index",
            "duplicate_canonical_text",
        ],
        how="left",
        validate="one_to_one",
    )

    retained_df = (
        manifest_df[
            manifest_df[
                "near_duplicate_status"
            ]
            == "retained"
        ]
        .copy()
        .reset_index(drop=True)
    )

    retained_df[
        "refined_unique_rank"
    ] = np.arange(
        1,
        len(retained_df) + 1,
    )

    return (
        retained_df,
        manifest_df,
    )


if ENABLE_NEAR_DUPLICATE_GATE:
    (
        near_unique_candidates_df,
        near_duplicate_manifest_df,
    ) = detect_near_duplicates(
        unique_candidates_df
    )

else:
    near_unique_candidates_df = (
        unique_candidates_df.copy()
    )

    near_unique_candidates_df[
        "near_duplicate_status"
    ] = "retained"

    near_unique_candidates_df[
        "duplicate_of_question_id"
    ] = None

    near_unique_candidates_df[
        "duplicate_detection_method"
    ] = None

    near_unique_candidates_df[
        "refined_unique_rank"
    ] = np.arange(
        1,
        len(
            near_unique_candidates_df
        ) + 1,
    )

    near_duplicate_manifest_df = (
        near_unique_candidates_df.copy()
    )


near_duplicates_removed = int(
    (
        near_duplicate_manifest_df[
            "near_duplicate_status"
        ]
        == "duplicate_removed"
    ).sum()
)

print(
    f"Exact duplicates removed earlier: "
    f"{duplicates_removed}"
)

print(
    f"Near duplicates removed in refinement: "
    f"{near_duplicates_removed}"
)

print(
    f"Candidates after refined deduplication: "
    f"{len(near_unique_candidates_df)}"
)

display(
    near_duplicate_manifest_df[
        [
            "question_id",
            "detected_topic",
            "official_reference",
            "marks",
            "near_duplicate_status",
            "duplicate_of_question_id",
            "duplicate_detection_method",
            "maximum_lexical_similarity",
            "maximum_token_jaccard",
            "maximum_minilm_similarity",
            "question_text",
        ]
    ].head(80)
)


## 10. Phase 2 final implementation — adaptive semantic and quality gate

This gate is applied after the original reranking and both duplicate-removal stages.

```text
refined unique candidates
        ↓
question-text quality analysis
        ↓
derive required candidate quota for each approved topic
        ↓
calculate one adaptive threshold per Agent 1 topic
        ↓
adaptive semantic gate
        ↓
quality-safe rescue only when a valid topic-balanced pool is still unavailable
        ↓
topic-distribution and marks balancing
```

### Previous approach

The first implementation tried:

```text
0.60 strict threshold
0.55 relaxed threshold
```

These values are preserved in the development markdown but are no longer active.

### Final adaptive approach

Each topic receives its own threshold using:

```text
60th percentile of its quality-safe scores
best score minus 0.10
absolute floor of 0.45
maximum threshold of 0.70
topic-specific minimum candidate quota
```

The threshold may be lowered within the safety range only when the topic needs more
candidates to satisfy the approved-topic and primary/supporting coverage requirements.

### Non-adaptive safety rules

```text
incomplete/noisy question text is always rejected
near duplicates remain removed
threshold never falls below the adaptive absolute floor
rescue candidates must still pass the text-quality gate
rescue candidates are clearly recorded and require human review
```

Every topic threshold and every candidate decision is exported for evaluation.


In [ ]:

INCOMPLETE_TRAILING_PATTERNS = [
    # Common exam-page/footer fragments observed across papers.
    r"\bdo not[.!]?$",
    r"\bturn over[.!]?$",
    r"\boutsid[.!]?$",
    r"\boutside[.!]?$",
    r"\bbo[.!]?$",
]

# Deliberately narrow and reusable. These tokens are highly
# likely to indicate an incomplete short final line when the
# line is not a question.
TRUNCATED_FINAL_TOKENS = {
    "and",
    "or",
    "the",
    "a",
    "an",
}

MAX_TRUNCATED_FINAL_LINE_WORDS = 8

ISOLATED_NOISE_LINES = {
    "do not",
    "turn over",
    "outside",
    "outsid",
    "bo",
    "do not write outside the box",
}


def is_non_instruction_label(
    segment: str,
) -> bool:
    cleaned = re.sub(
        r"\s+",
        " ",
        str(segment or "").strip(),
    )

    if not cleaned:
        return True

    label_patterns = [
        r"^(?:figure|table|diagram|flowchart)\s+\d+[a-z]?$",
        r"^\d+(?:\.\d+)+\s+[A-Za-z].*$",
        r"^[A-Za-z]\s+[A-Za-z]\s+[A-Za-z](?:\s+[A-Za-z])?$",
    ]

    return any(
        re.fullmatch(
            pattern,
            cleaned,
            flags=re.IGNORECASE,
        )
        for pattern in label_patterns
    )


def split_instruction_segments(
    raw_text: str,
) -> list[str]:
    """
    Build sentence-like segments from the full question text.

    This catches an incomplete instructional sentence even when a
    later label such as 'Figure 7' is the final line.
    """
    normalized = re.sub(
        r"\s+",
        " ",
        raw_text,
    ).strip()

    sentence_segments = re.split(
        r"(?<=[.!?])\s+",
        normalized,
    )

    raw_lines = [
        re.sub(
            r"\s+",
            " ",
            line.strip(),
        )
        for line in raw_text.splitlines()
        if line.strip()
    ]

    combined = []
    seen = set()

    for segment in [
        *sentence_segments,
        *raw_lines,
    ]:
        cleaned = segment.strip()

        if (
            not cleaned
            or is_non_instruction_label(
                cleaned
            )
        ):
            continue

        key = cleaned.lower()

        if key in seen:
            continue

        seen.add(key)
        combined.append(cleaned)

    return combined


def detect_question_quality_issues(
    question_text: Any,
) -> list[str]:
    raw_text = str(
        question_text
        or ""
    ).strip()

    if not raw_text:
        return [
            "missing_question_text"
        ]

    issues: list[str] = []

    normalized_text = re.sub(
        r"\s+",
        " ",
        raw_text,
    ).strip()

    word_count = len(
        re.findall(
            r"[A-Za-z0-9]+",
            normalized_text,
        )
    )

    if (
        word_count
        < MIN_QUESTION_WORD_COUNT
    ):
        issues.append(
            "question_text_too_short"
        )

    lowered = normalized_text.lower()

    if any(
        re.search(
            pattern,
            lowered,
        )
        for pattern in (
            INCOMPLETE_TRAILING_PATTERNS
        )
    ):
        issues.append(
            "incomplete_trailing_fragment"
        )

    suspicious_segments = []

    for segment in split_instruction_segments(
        raw_text
    ):
        segment_tokens = re.findall(
            r"[a-z0-9]+",
            segment.lower(),
        )

        if not segment_tokens:
            continue

        segment_is_question = (
            segment.rstrip()
            .endswith("?")
        )

        # Keep this deliberately narrow to avoid overfitting.
        # A sentence-like instructional segment ending in
        # "and." or "or." is highly likely to be truncated.
        dangling_connector = bool(
            len(segment_tokens) >= 4
            and segment_tokens[-1]
            in {
                "and",
                "or",
            }
            and not segment_is_question
            and re.search(
                r"[.!]\s*$",
                segment,
            )
        )

        if dangling_connector:
            suspicious_segments.append(
                segment
            )

    if suspicious_segments:
        issues.append(
            "probable_truncated_instruction_segment"
        )

    normalized_lines = [
        re.sub(
            r"\s+",
            " ",
            line.strip().lower(),
        )
        for line in raw_text.splitlines()
        if line.strip()
    ]

    if any(
        line in ISOLATED_NOISE_LINES
        for line in normalized_lines
    ):
        issues.append(
            "isolated_footer_or_instruction_fragment"
        )

    if (
        " outsid " in f" {lowered} "
        or " bo " in f" {lowered} "
    ):
        issues.append(
            "parser_noise_fragment"
        )

    return sorted(set(issues))


def derive_topic_candidate_requirements(
    topics_frame: pd.DataFrame,
) -> dict[int, int]:
    """
    Convert assessment-level requirements into a minimum
    candidate quota for each Agent 1 topic.

    Example for the current five-question request:

    primary tracing topic       -> 3 candidates
    supporting arrays topic     -> 1 candidate
    supporting iteration topic  -> 1 candidate
    """
    topic_rows = (
        topics_frame[
            [
                "agent1_topic_index",
                "role",
                "official_reference",
                "ranking_score",
                "role_weight",
            ]
        ]
        .drop_duplicates(
            subset=[
                "agent1_topic_index"
            ]
        )
        .copy()
    )

    topic_rows[
        "agent1_topic_index"
    ] = topic_rows[
        "agent1_topic_index"
    ].astype(int)

    requirements = {
        int(topic_index): 0
        for topic_index
        in topic_rows[
            "agent1_topic_index"
        ].tolist()
    }

    required_references = set(
        request.get(
            "required_official_references",
            [],
        )
    )

    # Every explicitly required approved reference receives
    # at least one candidate.
    for _, topic_row in (
        topic_rows.iterrows()
    ):
        if (
            str(
                topic_row[
                    "official_reference"
                ]
            )
            in required_references
        ):
            requirements[
                int(
                    topic_row[
                        "agent1_topic_index"
                    ]
                )
            ] = 1

    def distribute_role_requirement(
        role: str,
        required_total: int,
    ) -> None:
        role_rows = (
            topic_rows[
                topic_rows[
                    "role"
                ]
                == role
            ]
            .sort_values(
                [
                    "ranking_score",
                    "agent1_topic_index",
                ],
                ascending=[
                    False,
                    True,
                ],
            )
        )

        if role_rows.empty:
            return

        role_indexes = [
            int(value)
            for value in role_rows[
                "agent1_topic_index"
            ].tolist()
        ]

        already_allocated = sum(
            requirements[index]
            for index in role_indexes
        )

        remaining = max(
            0,
            int(required_total)
            - already_allocated,
        )

        position = 0

        while remaining > 0:
            topic_index = role_indexes[
                position
                % len(role_indexes)
            ]

            requirements[
                topic_index
            ] += 1

            remaining -= 1
            position += 1

    distribute_role_requirement(
        "primary",
        request[
            "minimum_primary_questions"
        ],
    )

    distribute_role_requirement(
        "supporting",
        request[
            "minimum_supporting_questions"
        ],
    )

    # Allocate any remaining assessment slots according to
    # Agent 1 role and ranking priority.
    remaining_total = max(
        0,
        int(
            request[
                "number_of_questions"
            ]
        )
        - sum(
            requirements.values()
        ),
    )

    priority_rows = (
        topic_rows.sort_values(
            [
                "role_weight",
                "ranking_score",
                "agent1_topic_index",
            ],
            ascending=[
                False,
                False,
                True,
            ],
        )
    )

    priority_indexes = [
        int(value)
        for value in priority_rows[
            "agent1_topic_index"
        ].tolist()
    ]

    position = 0

    while (
        remaining_total > 0
        and priority_indexes
    ):
        topic_index = priority_indexes[
            position
            % len(priority_indexes)
        ]

        requirements[
            topic_index
        ] += 1

        remaining_total -= 1
        position += 1

    return requirements


def calculate_adaptive_threshold_profile(
    candidates: pd.DataFrame,
) -> tuple[
    pd.DataFrame,
    pd.DataFrame,
]:
    """
    Calculate one semantic threshold for each approved
    Agent 1 topic using its own quality-safe score
    distribution and required candidate quota.
    """
    working = candidates.copy()

    working[
        "question_quality_issues"
    ] = working[
        "question_text"
    ].map(
        detect_question_quality_issues
    )

    working[
        "question_quality_gate_passed"
    ] = working[
        "question_quality_issues"
    ].map(
        lambda issues: (
            len(issues) == 0
        )
    )

    topic_requirements = (
        derive_topic_candidate_requirements(
            validated_topics_df
        )
    )

    profile_rows: list[
        dict[str, Any]
    ] = []

    for _, topic_row in (
        validated_topics_df
        .drop_duplicates(
            subset=[
                "agent1_topic_index"
            ]
        )
        .sort_values(
            "agent1_topic_index"
        )
        .iterrows()
    ):
        topic_index = int(
            topic_row[
                "agent1_topic_index"
            ]
        )

        topic_candidates = working[
            working[
                "agent1_topic_index"
            ]
            == topic_index
        ]

        quality_safe = topic_candidates[
            topic_candidates[
                "question_quality_gate_passed"
            ]
        ]

        scores = (
            quality_safe[
                "semantic_score"
            ]
            .astype(float)
            .sort_values(
                ascending=False
            )
            .to_numpy()
        )

        required_count = int(
            topic_requirements.get(
                topic_index,
                0,
            )
        )

        if len(scores) == 0:
            profile_rows.append(
                {
                    "agent1_topic_index": (
                        topic_index
                    ),
                    "detected_topic": (
                        topic_row[
                            "detected_topic"
                        ]
                    ),
                    "role": (
                        topic_row["role"]
                    ),
                    "official_reference": (
                        topic_row[
                            "official_reference"
                        ]
                    ),
                    "candidate_count": int(
                        len(topic_candidates)
                    ),
                    "quality_safe_candidate_count": 0,
                    "required_candidate_quota": (
                        required_count
                    ),
                    "minimum_score": None,
                    "maximum_score": None,
                    "median_score": None,
                    "percentile_score": None,
                    "top_score_margin_threshold": None,
                    "distribution_threshold": (
                        ADAPTIVE_MAXIMUM_ALLOWED_THRESHOLD
                    ),
                    "adaptive_threshold": (
                        ADAPTIVE_MAXIMUM_ALLOWED_THRESHOLD
                    ),
                    "candidates_passing_threshold": 0,
                    "quota_adjustment_used": False,
                    "quota_satisfied_above_floor": False,
                    "threshold_reason": (
                        "no_quality_safe_candidates"
                    ),
                }
            )

            continue

        minimum_score = float(
            np.min(scores)
        )

        maximum_score = float(
            np.max(scores)
        )

        median_score = float(
            np.median(scores)
        )

        percentile_score = float(
            np.percentile(
                scores,
                ADAPTIVE_SCORE_PERCENTILE,
            )
        )

        top_margin_threshold = float(
            maximum_score
            - ADAPTIVE_TOP_SCORE_MARGIN
        )

        unbounded_threshold = max(
            percentile_score,
            top_margin_threshold,
        )

        distribution_threshold = float(
            np.clip(
                unbounded_threshold,
                ADAPTIVE_ABSOLUTE_MINIMUM_SCORE,
                ADAPTIVE_MAXIMUM_ALLOWED_THRESHOLD,
            )
        )

        adaptive_threshold = (
            distribution_threshold
        )

        quota_adjustment_used = False
        threshold_reason = (
            "topic_score_distribution"
        )

        passing_count = int(
            (
                scores
                + ADAPTIVE_THRESHOLD_EPSILON
                >= adaptive_threshold
            ).sum()
        )

        if (
            required_count > 0
            and passing_count
            < required_count
            and len(scores)
            >= required_count
        ):
            quota_boundary_score = float(
                scores[
                    required_count - 1
                ]
            )

            if (
                quota_boundary_score
                >= ADAPTIVE_ABSOLUTE_MINIMUM_SCORE
            ):
                adaptive_threshold = max(
                    ADAPTIVE_ABSOLUTE_MINIMUM_SCORE,
                    min(
                        distribution_threshold,
                        quota_boundary_score,
                    ),
                )

                quota_adjustment_used = True
                threshold_reason = (
                    "lowered_within_safety_range_"
                    "for_topic_quota"
                )

                passing_count = int(
                    (
                        scores
                        + ADAPTIVE_THRESHOLD_EPSILON
                        >= adaptive_threshold
                    ).sum()
                )

            else:
                threshold_reason = (
                    "topic_quota_requires_score_"
                    "below_adaptive_floor"
                )

        quota_satisfied = bool(
            passing_count
            >= required_count
        )

        profile_rows.append(
            {
                "agent1_topic_index": (
                    topic_index
                ),
                "detected_topic": (
                    topic_row[
                        "detected_topic"
                    ]
                ),
                "role": (
                    topic_row["role"]
                ),
                "official_reference": (
                    topic_row[
                        "official_reference"
                    ]
                ),
                "candidate_count": int(
                    len(topic_candidates)
                ),
                "quality_safe_candidate_count": int(
                    len(quality_safe)
                ),
                "required_candidate_quota": (
                    required_count
                ),
                "minimum_score": round(
                    minimum_score,
                    6,
                ),
                "maximum_score": round(
                    maximum_score,
                    6,
                ),
                "median_score": round(
                    median_score,
                    6,
                ),
                "percentile_score": round(
                    percentile_score,
                    6,
                ),
                "top_score_margin_threshold": round(
                    top_margin_threshold,
                    6,
                ),
                "distribution_threshold": round(
                    distribution_threshold,
                    6,
                ),
                "adaptive_threshold": round(
                    adaptive_threshold,
                    6,
                ),
                "candidates_passing_threshold": (
                    passing_count
                ),
                "quota_adjustment_used": (
                    quota_adjustment_used
                ),
                "quota_satisfied_above_floor": (
                    quota_satisfied
                ),
                "threshold_reason": (
                    threshold_reason
                ),
            }
        )

    profile_df = pd.DataFrame(
        profile_rows
    )

    threshold_lookup = (
        profile_df[
            [
                "agent1_topic_index",
                "adaptive_threshold",
                "required_candidate_quota",
                "threshold_reason",
            ]
        ]
        .copy()
    )

    gated = working.merge(
        threshold_lookup,
        on="agent1_topic_index",
        how="left",
        validate="many_to_one",
    )

    if gated[
        "adaptive_threshold"
    ].isna().any():
        raise RuntimeError(
            "An adaptive threshold could not be assigned "
            "to every candidate."
        )

    gated[
        "phase2_semantic_threshold"
    ] = gated[
        "adaptive_threshold"
    ].astype(float)

    gated[
        "concept_gate_passed"
    ] = (
        gated[
            "semantic_score"
        ]
        + ADAPTIVE_THRESHOLD_EPSILON
        >= gated[
            "phase2_semantic_threshold"
        ]
    )

    gated[
        "phase2_gate_passed"
    ] = (
        gated[
            "concept_gate_passed"
        ]
        & gated[
            "question_quality_gate_passed"
        ]
    )

    gated[
        "phase2_gate_mode"
    ] = np.where(
        gated[
            "phase2_gate_passed"
        ],
        "adaptive_threshold",
        "rejected",
    )

    gated[
        "semantic_rescue_used"
    ] = False

    def rejection_reasons(
        row: pd.Series,
    ) -> list[str]:
        reasons: list[str] = []

        if not bool(
            row[
                "concept_gate_passed"
            ]
        ):
            reasons.append(
                "semantic_score_below_"
                "topic_adaptive_threshold"
            )

        reasons.extend(
            row[
                "question_quality_issues"
            ]
        )

        return sorted(set(reasons))

    gated[
        "phase2_rejection_reasons"
    ] = gated.apply(
        rejection_reasons,
        axis=1,
    )

    return gated, profile_df


def pool_is_sufficient(
    gated: pd.DataFrame,
) -> bool:
    passed = gated[
        gated[
            "phase2_gate_passed"
        ]
    ]

    return bool(
        len(passed)
        >= request[
            "number_of_questions"
        ]
        and (
            passed[
                "agent1_role"
            ]
            == "primary"
        ).sum()
        >= request[
            "minimum_primary_questions"
        ]
        and (
            passed[
                "agent1_role"
            ]
            == "supporting"
        ).sum()
        >= request[
            "minimum_supporting_questions"
        ]
        and passed[
            "official_reference"
        ].nunique()
        >= request[
            "minimum_distinct_official_references"
        ]
        and set(
            request.get(
                "required_official_references",
                [],
            )
        ).issubset(
            set(
                passed[
                    "official_reference"
                ]
                .dropna()
                .astype(str)
                .tolist()
            )
        )
    )


def build_quality_safe_rescue_pool(
    adaptive_gated: pd.DataFrame,
) -> pd.DataFrame:
    """
    Retain the adaptive gate and add only the strongest
    quality-safe below-threshold candidates needed to satisfy
    required topic, role and total-count constraints.

    This is not adaptive-threshold relaxation. Every rescued
    record is explicitly labelled for human review.
    """
    rescued = adaptive_gated.copy()

    quality_safe_mask = (
        rescued[
            "question_quality_gate_passed"
        ]
        & (
            rescued[
                "semantic_score"
            ]
            >= MIN_TOPIC_RESCUE_SEMANTIC_SCORE
        )
    )

    required_references = set(
        request.get(
            "required_official_references",
            [],
        )
    )

    def passed_rows() -> pd.DataFrame:
        return rescued[
            rescued[
                "phase2_gate_passed"
            ]
        ]

    # A. Required approved references
    currently_covered = set(
        passed_rows()[
            "official_reference"
        ]
        .dropna()
        .astype(str)
        .tolist()
    )

    missing_references = sorted(
        required_references
        - currently_covered
    )

    for reference in missing_references:
        options = (
            rescued[
                quality_safe_mask
                & (
                    rescued[
                        "official_reference"
                    ].astype(str)
                    == reference
                )
                & ~rescued[
                    "phase2_gate_passed"
                ]
            ]
            .sort_values(
                [
                    "semantic_score",
                    "final_score",
                ],
                ascending=False,
            )
        )

        if options.empty:
            continue

        selected_index = (
            options.index[0]
        )

        rescued.loc[
            selected_index,
            "phase2_gate_passed",
        ] = True

        rescued.loc[
            selected_index,
            "phase2_gate_mode",
        ] = (
            "quality_safe_topic_rescue"
        )

        rescued.loc[
            selected_index,
            "semantic_rescue_used",
        ] = True

        rescued.at[
            selected_index,
            "phase2_rejection_reasons",
        ] = [
            "adaptive_threshold_rescued_"
            "for_topic_coverage"
        ]

    # B. Role minimums
    role_requirements = {
        "primary": request[
            "minimum_primary_questions"
        ],
        "supporting": request[
            "minimum_supporting_questions"
        ],
    }

    for role, required_count in (
        role_requirements.items()
    ):
        current_count = int(
            (
                passed_rows()[
                    "agent1_role"
                ]
                == role
            ).sum()
        )

        needed = max(
            0,
            int(required_count)
            - current_count,
        )

        if needed == 0:
            continue

        options = (
            rescued[
                quality_safe_mask
                & (
                    rescued[
                        "agent1_role"
                    ]
                    == role
                )
                & ~rescued[
                    "phase2_gate_passed"
                ]
            ]
            .sort_values(
                [
                    "semantic_score",
                    "final_score",
                ],
                ascending=False,
            )
        )

        for selected_index in (
            options.head(needed).index
        ):
            rescued.loc[
                selected_index,
                "phase2_gate_passed",
            ] = True

            rescued.loc[
                selected_index,
                "phase2_gate_mode",
            ] = (
                "quality_safe_role_rescue"
            )

            rescued.loc[
                selected_index,
                "semantic_rescue_used",
            ] = True

            rescued.at[
                selected_index,
                "phase2_rejection_reasons",
            ] = [
                "adaptive_threshold_rescued_"
                "for_role_coverage"
            ]

    # C. Total candidate count
    additional_needed = max(
        0,
        int(
            request[
                "number_of_questions"
            ]
        )
        - int(
            rescued[
                "phase2_gate_passed"
            ].sum()
        ),
    )

    if additional_needed > 0:
        options = (
            rescued[
                quality_safe_mask
                & ~rescued[
                    "phase2_gate_passed"
                ]
            ]
            .sort_values(
                [
                    "semantic_score",
                    "final_score",
                ],
                ascending=False,
            )
        )

        for selected_index in (
            options.head(
                additional_needed
            ).index
        ):
            rescued.loc[
                selected_index,
                "phase2_gate_passed",
            ] = True

            rescued.loc[
                selected_index,
                "phase2_gate_mode",
            ] = (
                "quality_safe_count_rescue"
            )

            rescued.loc[
                selected_index,
                "semantic_rescue_used",
            ] = True

            rescued.at[
                selected_index,
                "phase2_rejection_reasons",
            ] = [
                "adaptive_threshold_rescued_"
                "for_candidate_count"
            ]

    return rescued


if ENABLE_PHASE2_GATE:
    if not ENABLE_ADAPTIVE_SEMANTIC_THRESHOLD:
        raise RuntimeError(
            "The final Notebook 05 configuration requires "
            "adaptive semantic thresholding."
        )

    (
        adaptive_gate_df,
        adaptive_threshold_profile_df,
    ) = calculate_adaptive_threshold_profile(
        near_unique_candidates_df
    )

    adaptive_pool_sufficient = (
        pool_is_sufficient(
            adaptive_gate_df
        )
    )

    if adaptive_pool_sufficient:
        phase2_gate_manifest_df = (
            adaptive_gate_df
        )

        phase2_rescue_used = False

    elif ENABLE_QUALITY_SAFE_TOPIC_RESCUE:
        rescue_gate_df = (
            build_quality_safe_rescue_pool(
                adaptive_gate_df
            )
        )

        if not pool_is_sufficient(
            rescue_gate_df
        ):
            diagnostics = {
                "requested_questions": (
                    request[
                        "number_of_questions"
                    ]
                ),
                "required_references": (
                    request.get(
                        "required_official_references",
                        [],
                    )
                ),
                "adaptive_quality_safe_candidates": int(
                    adaptive_gate_df[
                        "question_quality_gate_passed"
                    ].sum()
                ),
                "adaptive_passing_candidates": int(
                    adaptive_gate_df[
                        "phase2_gate_passed"
                    ].sum()
                ),
                "quality_safe_candidates_above_rescue_floor": int(
                    (
                        adaptive_gate_df[
                            "question_quality_gate_passed"
                        ]
                        & (
                            adaptive_gate_df[
                                "semantic_score"
                            ]
                            >= MIN_TOPIC_RESCUE_SEMANTIC_SCORE
                        )
                    ).sum()
                ),
                "rescue_floor": (
                    MIN_TOPIC_RESCUE_SEMANTIC_SCORE
                ),
            }

            display(
                pd.DataFrame(
                    [diagnostics]
                )
            )

            raise RuntimeError(
                "No sufficient quality-safe candidate pool "
                "exists after adaptive thresholding. "
                "Review Agent 1 topics or assessment "
                "requirements."
            )

        phase2_gate_manifest_df = (
            rescue_gate_df
        )

        phase2_rescue_used = bool(
            rescue_gate_df[
                "semantic_rescue_used"
            ].any()
        )

    else:
        display(
            adaptive_threshold_profile_df
        )

        raise RuntimeError(
            "The adaptive candidate pool cannot satisfy "
            "the requested topic coverage."
        )

else:
    phase2_gate_manifest_df = (
        near_unique_candidates_df.copy()
    )

    phase2_gate_manifest_df[
        "question_quality_issues"
    ] = [
        []
        for _ in range(
            len(
                phase2_gate_manifest_df
            )
        )
    ]

    phase2_gate_manifest_df[
        "question_quality_gate_passed"
    ] = True

    phase2_gate_manifest_df[
        "adaptive_threshold"
    ] = None

    phase2_gate_manifest_df[
        "phase2_semantic_threshold"
    ] = None

    phase2_gate_manifest_df[
        "concept_gate_passed"
    ] = True

    phase2_gate_manifest_df[
        "phase2_gate_passed"
    ] = True

    phase2_gate_manifest_df[
        "phase2_gate_mode"
    ] = "disabled"

    phase2_gate_manifest_df[
        "semantic_rescue_used"
    ] = False

    phase2_gate_manifest_df[
        "phase2_rejection_reasons"
    ] = [
        []
        for _ in range(
            len(
                phase2_gate_manifest_df
            )
        )
    ]

    adaptive_threshold_profile_df = (
        pd.DataFrame()
    )

    adaptive_pool_sufficient = True
    phase2_rescue_used = False


phase2_candidates_df = (
    phase2_gate_manifest_df[
        phase2_gate_manifest_df[
            "phase2_gate_passed"
        ]
    ]
    .sort_values(
        [
            "final_score",
            "semantic_score",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)

phase2_candidates_df[
    "phase2_rank"
] = np.arange(
    1,
    len(
        phase2_candidates_df
    ) + 1,
)

phase2_gate_manifest_df = (
    phase2_gate_manifest_df.merge(
        phase2_candidates_df[
            [
                "question_id",
                "agent1_topic_index",
                "phase2_rank",
            ]
        ],
        on=[
            "question_id",
            "agent1_topic_index",
        ],
        how="left",
        validate="one_to_one",
    )
)

phase2_rejected_df = (
    phase2_gate_manifest_df[
        ~phase2_gate_manifest_df[
            "phase2_gate_passed"
        ]
    ]
    .copy()
)


adaptive_threshold_values = (
    adaptive_threshold_profile_df[
        "adaptive_threshold"
    ]
    .dropna()
    .astype(float)
    if not adaptive_threshold_profile_df.empty
    else pd.Series(
        dtype=float
    )
)

adaptive_threshold_min = (
    float(
        adaptive_threshold_values.min()
    )
    if not adaptive_threshold_values.empty
    else None
)

adaptive_threshold_max = (
    float(
        adaptive_threshold_values.max()
    )
    if not adaptive_threshold_values.empty
    else None
)

adaptive_threshold_mean = (
    float(
        adaptive_threshold_values.mean()
    )
    if not adaptive_threshold_values.empty
    else None
)


phase2_gate_summary = {
    "phase2_version": PHASE2_VERSION,
    "threshold_strategy": (
        "per_topic_adaptive"
    ),
    "adaptive_threshold_version": (
        ADAPTIVE_THRESHOLD_VERSION
    ),
    "legacy_fixed_strict_threshold": (
        LEGACY_FIXED_STRICT_THRESHOLD
    ),
    "legacy_fixed_relaxed_threshold": (
        LEGACY_FIXED_RELAXED_THRESHOLD
    ),
    "legacy_fixed_thresholds_active": (
        LEGACY_FIXED_THRESHOLD_APPROACH_ENABLED
    ),
    "adaptive_score_percentile": (
        ADAPTIVE_SCORE_PERCENTILE
    ),
    "adaptive_top_score_margin": (
        ADAPTIVE_TOP_SCORE_MARGIN
    ),
    "adaptive_absolute_floor": (
        ADAPTIVE_ABSOLUTE_MINIMUM_SCORE
    ),
    "adaptive_threshold_cap": (
        ADAPTIVE_MAXIMUM_ALLOWED_THRESHOLD
    ),
    "adaptive_threshold_min": (
        adaptive_threshold_min
    ),
    "adaptive_threshold_max": (
        adaptive_threshold_max
    ),
    "adaptive_threshold_mean": (
        adaptive_threshold_mean
    ),
    "adaptive_pool_sufficient_without_rescue": (
        adaptive_pool_sufficient
    ),
    "quality_safe_rescue_enabled": (
        ENABLE_QUALITY_SAFE_TOPIC_RESCUE
    ),
    "quality_safe_rescue_used": (
        phase2_rescue_used
    ),
    "rescue_semantic_floor": (
        MIN_TOPIC_RESCUE_SEMANTIC_SCORE
    ),
    "rescued_candidate_count": int(
        phase2_gate_manifest_df[
            "semantic_rescue_used"
        ].sum()
    ),
    "baseline_unique_candidates": int(
        len(
            unique_candidates_df
        )
    ),
    "near_duplicates_removed": (
        near_duplicates_removed
    ),
    "refined_unique_candidates": int(
        len(
            near_unique_candidates_df
        )
    ),
    "phase2_eligible_candidates": int(
        len(
            phase2_candidates_df
        )
    ),
    "phase2_rejected_candidates": int(
        len(
            phase2_rejected_df
        )
    ),
    "semantic_gate_rejections": int(
        (
            ~phase2_gate_manifest_df[
                "concept_gate_passed"
            ]
        ).sum()
    ),
    "question_quality_rejections": int(
        (
            ~phase2_gate_manifest_df[
                "question_quality_gate_passed"
            ]
        ).sum()
    ),
    "topics_using_chunk_evidence": int(
        len(
            validated_topics_df
        )
        - fallback_evidence_topic_count
    ),
    "topics_using_summary_fallback": (
        fallback_evidence_topic_count
    ),
}


print(
    "Adaptive threshold profile:"
)

display(
    adaptive_threshold_profile_df
)

print(
    "Phase 2 gate summary:"
)

display(
    pd.DataFrame(
        [phase2_gate_summary]
    )
)

display(
    phase2_gate_manifest_df[
        [
            "unique_rank",
            "phase2_rank",
            "detected_topic",
            "agent1_role",
            "official_reference",
            "marks",
            "semantic_score",
            "phase2_semantic_threshold",
            "threshold_reason",
            "concept_gate_passed",
            "question_quality_gate_passed",
            "phase2_gate_passed",
            "phase2_gate_mode",
            "semantic_rescue_used",
            "question_quality_issues",
            "phase2_rejection_reasons",
            "question_text",
        ]
    ].head(100)
)

if not phase2_rejected_df.empty:
    print(
        "Rejected candidate examples:"
    )

    display(
        phase2_rejected_df[
            [
                "detected_topic",
                "official_reference",
                "semantic_score",
                "phase2_semantic_threshold",
                "threshold_reason",
                "question_quality_issues",
                "phase2_rejection_reasons",
                "question_text",
            ]
        ].head(30)
    )


## 10. Select a balanced question set

Dynamic programming selects:

- the requested number of questions;
- at least the requested number from primary topics where possible;
- total marks closest to the target;
- the strongest combined retrieval score.


In [ ]:

def select_questions(
    candidates: pd.DataFrame,
) -> tuple[pd.DataFrame, dict[str, Any]]:
    n_questions = request["number_of_questions"]
    target_marks = request["target_total_marks"]
    minimum_primary = request[
        "minimum_primary_questions"
    ]
    minimum_supporting = request[
        "minimum_supporting_questions"
    ]
    minimum_distinct = request[
        "minimum_distinct_official_references"
    ]

    required_references = set(
        request.get(
            "required_official_references",
            [],
        )
    )

    if len(candidates) < n_questions:
        raise RuntimeError(
            "Not enough Phase 2 eligible candidates."
        )

    search_limit = min(
        max(n_questions * 14, 50),
        len(candidates),
    )

    working = (
        candidates.head(search_limit)
        .reset_index(drop=True)
    )

    references = sorted(
        working["official_reference"]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )

    ref_bits = {
        reference: 1 << index
        for index, reference
        in enumerate(references)
    }

    # State:
    # count, marks, primary, supporting, reference_mask
    states = {
        (0, 0, 0, 0, 0): (
            0.0,
            tuple(),
        )
    }

    for index, row in working.iterrows():
        next_states = dict(states)

        for state, value in states.items():
            (
                count,
                marks,
                primary,
                supporting,
                reference_mask,
            ) = state

            score, selected = value

            if count >= n_questions:
                continue

            new_state = (
                count + 1,
                marks + int(row["marks"]),
                primary
                + int(
                    row["agent1_role"] == "primary"
                ),
                supporting
                + int(
                    row["agent1_role"] == "supporting"
                ),
                reference_mask
                | ref_bits[
                    str(row["official_reference"])
                ],
            )

            new_value = (
                score + float(row["final_score"]),
                selected + (index,),
            )

            previous = next_states.get(new_state)

            if (
                previous is None
                or new_value[0] > previous[0]
            ):
                next_states[new_state] = new_value

        states = next_states


    options = []

    for state, value in states.items():
        (
            count,
            marks,
            primary,
            supporting,
            reference_mask,
        ) = state

        score, selected = value

        if count != n_questions:
            continue

        distinct_count = int(
            reference_mask.bit_count()
        )

        selected_references = {
            reference
            for reference, bit_value
            in ref_bits.items()
            if reference_mask
            & bit_value
        }

        requirements_met = bool(
            primary >= minimum_primary
            and supporting >= minimum_supporting
            and distinct_count >= minimum_distinct
            and required_references.issubset(
                selected_references
            )
        )

        options.append(
            {
                "marks": marks,
                "primary_count": primary,
                "supporting_count": supporting,
                "distinct_reference_count": (
                    distinct_count
                ),
                "score": score,
                "selected": selected,
                "marks_difference": abs(
                    marks - target_marks
                ),
                "requirements_met": (
                    requirements_met
                ),
            }
        )


    options_df = pd.DataFrame(options)

    if options_df.empty:
        raise RuntimeError(
            "No question combination was generated."
        )

    eligible_options = options_df[
        options_df["requirements_met"]
    ]

    if eligible_options.empty:
        display(
            options_df.sort_values(
                ["marks_difference", "score"],
                ascending=[True, False],
            ).head(30)
        )
        raise RuntimeError(
            "No final combination satisfies the "
            "Phase 2 topic-coverage requirements."
        )

    best = (
        eligible_options.sort_values(
            [
                "marks_difference",
                "score",
                "distinct_reference_count",
            ],
            ascending=[True, False, False],
        )
        .iloc[0]
    )

    selected_df = (
        working.loc[
            list(best["selected"])
        ]
        .sort_values(
            ["agent1_role", "final_score"],
            ascending=[True, False],
        )
        .reset_index(drop=True)
    )

    selected_df["selected_rank"] = np.arange(
        1,
        len(selected_df) + 1,
    )

    selected_marks = int(
        selected_df["marks"].sum()
    )

    marks_difference = abs(
        selected_marks
        - target_marks
    )

    marks_within_tolerance = bool(
        marks_difference
        <= TARGET_MARKS_TOLERANCE
    )

    semantic_rescue_selected = bool(
        selected_df[
            "semantic_rescue_used"
        ].any()
    )

    assessment_release_status = (
        "provisional_evaluation_ready"
        if (
            marks_within_tolerance
            and not semantic_rescue_selected
        )
        else "needs_user_decision"
    )

    if (
        not marks_within_tolerance
        and not ALLOW_BEST_AVAILABLE_ASSESSMENT
    ):
        raise RuntimeError(
            "The best quality-safe assessment is outside "
            "the permitted target-marks tolerance."
        )

    summary = {
        "requested_questions": n_questions,
        "selected_questions": int(
            len(selected_df)
        ),
        "target_marks": target_marks,
        "selected_marks": selected_marks,
        "marks_difference": marks_difference,
        "target_marks_tolerance": (
            TARGET_MARKS_TOLERANCE
        ),
        "marks_within_tolerance": (
            marks_within_tolerance
        ),
        "assessment_release_status": (
            assessment_release_status
        ),
        "semantic_rescue_selected": (
            semantic_rescue_selected
        ),
        "requires_user_decision": (
            (
                not marks_within_tolerance
            )
            or semantic_rescue_selected
        ),
        "minimum_primary_questions": (
            minimum_primary
        ),
        "selected_primary_questions": int(
            (
                selected_df["agent1_role"]
                == "primary"
            ).sum()
        ),
        "minimum_supporting_questions": (
            minimum_supporting
        ),
        "selected_supporting_questions": int(
            (
                selected_df["agent1_role"]
                == "supporting"
            ).sum()
        ),
        "minimum_distinct_official_references": (
            minimum_distinct
        ),
        "selected_distinct_official_references": int(
            selected_df[
                "official_reference"
            ].nunique()
        ),
        "selected_official_references": sorted(
            selected_df[
                "official_reference"
            ]
            .dropna()
            .astype(str)
            .unique()
            .tolist()
        ),
        "required_official_references": sorted(
            required_references
        ),
        "all_required_references_covered": (
            required_references.issubset(
                set(
                    selected_df[
                        "official_reference"
                    ]
                    .dropna()
                    .astype(str)
                    .tolist()
                )
            )
        ),
        "fallback_selected": int(
            (
                selected_df["retrieval_stage"]
                == "same_section_fallback"
            ).sum()
        ),
        "phase2_threshold_strategy": (
            "per_topic_adaptive"
        ),
        "adaptive_threshold_min": (
            adaptive_threshold_min
        ),
        "adaptive_threshold_max": (
            adaptive_threshold_max
        ),
        "adaptive_threshold_mean": (
            adaptive_threshold_mean
        ),
        "adaptive_pool_sufficient_without_rescue": (
            adaptive_pool_sufficient
        ),
        "phase2_rescue_used": (
            phase2_rescue_used
        ),
    }

    return selected_df, summary


(
    selected_candidates_df,
    selection_summary,
) = select_questions(
    phase2_candidates_df
)

display(pd.DataFrame([selection_summary]))

display(
    selected_candidates_df[
        [
            "selected_rank",
            "phase2_rank",
            "detected_topic",
            "agent1_role",
            "retrieval_stage",
            "official_reference",
            "question_number",
            "marks",
            "semantic_score",
            "phase2_semantic_threshold",
            "question_quality_issues",
            "final_score",
            "question_text",
        ]
    ]
)


if selection_summary[
    "assessment_release_status"
] == "needs_user_decision":
    print(
        "\nASSESSMENT RELEASE WARNING"
    )
    print(
        "The strongest quality-safe result is outside "
        "the configured target-marks tolerance."
    )
    print(
        f"Requested marks: "
        f"{selection_summary['target_marks']}"
    )
    print(
        f"Selected marks: "
        f"{selection_summary['selected_marks']}"
    )
    print(
        "Review the assessment before releasing it to "
        "a student. Possible actions include changing "
        "the question count, marks target or approved "
        "topic distribution."
    )

    if selection_summary[
        "semantic_rescue_selected"
    ]:
        print(
            "One or more selected questions were included "
            "through the documented quality-safe semantic "
            "rescue. Human review is required."
        )


## 11. Fetch full questions and mark schemes from PostgreSQL

Qdrant provides IDs and ranking metadata. PostgreSQL provides the complete
source-of-truth assessment records.


In [ ]:
selected_ids = (
    selected_candidates_df[
        "question_id"
    ]
    .astype(str)
    .tolist()
)

selected_uuids = [
    uuid.UUID(value)
    for value in selected_ids
]


bundle_query = (
    select(
        questions.c.id.label(
            "question_id"
        ),
        questions.c.question_uid,
        questions.c.question_number,
        questions.c.question_text,
        questions.c.context_text,
        questions.c.marks,
        questions.c.page_start,
        questions.c.page_end,
        questions.c.has_code,
        questions.c.has_visual,
        questions.c.visual_page_numbers,
        questions.c.review_status,

        QUESTION_DOCUMENT_FK_COLUMN.label(
            "question_document_id"
        ),
        DOCUMENT_PATH_COLUMN.label(
            "source_pdf_path"
        ),
        (
            DOCUMENT_FILE_NAME_COLUMN
            if DOCUMENT_FILE_NAME_COLUMN is not None
            else DOCUMENT_PATH_COLUMN
        ).label(
            "source_file_name"
        ),
        (
            DOCUMENT_SOURCE_URL_COLUMN
            if DOCUMENT_SOURCE_URL_COLUMN is not None
            else DOCUMENT_PATH_COLUMN
        ).label(
            "source_document_url"
        ),

        questions.c.official_reference,
        questions.c.official_concept_name,
        questions.c.official_section_reference,
        questions.c.official_section_name,

        topics.c.pmt_subtopic_code,
        topics.c.pmt_subtopic_name,
        topics.c.paper_code,
        topics.c.programming_language,

        question_ms_links.c.match_method,
        question_ms_links.c.match_confidence,

        mark_schemes.c.id.label(
            "mark_scheme_id"
        ),
        mark_schemes.c.mark_scheme_uid,
        mark_schemes.c.maximum_marks,
        mark_schemes.c.marking_guidance,
        mark_schemes.c.marking_points,
        mark_schemes.c.acceptable_answers,
        mark_schemes.c.rejected_answers,
        mark_schemes.c.additional_guidance,
        mark_schemes.c.assessment_objectives,
    )
    .select_from(
        questions
        .join(
            topics,
            questions.c.topic_id
            == topics.c.id,
        )
        .join(
            documents,
            documents.c.id
            == QUESTION_DOCUMENT_FK_COLUMN,
        )
        .join(
            question_ms_links,
            question_ms_links.c.question_id
            == questions.c.id,
        )
        .join(
            mark_schemes,
            mark_schemes.c.id
            == question_ms_links
            .c.mark_scheme_entry_id,
        )
    )
    .where(
        questions.c.id.in_(
            selected_uuids
        )
    )
)

with engine.connect() as connection:
    bundles_df = pd.read_sql(
        bundle_query,
        connection,
    )

bundles_df["question_id"] = (
    bundles_df[
        "question_id"
    ].astype(str)
)

final_df = (
    selected_candidates_df.merge(
        bundles_df,
        on="question_id",
        how="left",
        suffixes=(
            "_retrieval",
            "_postgres",
        ),
        validate="one_to_one",
    )
    .sort_values(
        "selected_rank"
    )
    .reset_index(drop=True)
)

required_merged_columns = [
    "source_pdf_path",
    "question_document_id",
    "paper_code_postgres",
    "programming_language_postgres",
    "question_number_postgres",
    "question_text_postgres",
    "marks_postgres",
    "has_code_postgres",
    "has_visual_postgres",
    "official_reference_postgres",
    "official_concept_name_postgres",
]

missing_merged_columns = [
    column
    for column in required_merged_columns
    if column not in final_df.columns
]

if missing_merged_columns:
    raise RuntimeError(
        "Expected merged columns are missing: "
        f"{missing_merged_columns}"
    )

if final_df["mark_scheme_id"].isna().any():
    raise RuntimeError(
        "A selected question has no mark scheme."
    )

print(
    f"Complete QP/MS bundles: "
    f"{len(final_df)}"
)

display(
    final_df[
        [
            "selected_rank",
            "detected_topic",
            "agent1_role",
            "official_reference_postgres",
            "question_number_postgres",
            "marks_postgres",
            "question_text_postgres",
            "maximum_marks",
            "marking_guidance",
        ]
    ]
)


## 12. Phase 1 implementation — render original visual-question pages

This section is added **after** the original retrieval and PostgreSQL QP/MS-fetch
logic. The earlier pipeline remains unchanged and visible above.

### Rendering rule

```text
has_visual = False
→ no page image is required

has_visual = True
→ render visual_page_numbers when available
→ otherwise render page_start through page_end
```

### Source used

The image is rendered from the original cached **Question Paper PDF** registered in
Notebook 01 and linked to the question record during parsing.

### Output location

```text
Agent2/OUTPUT/visual_question_pages/<run timestamp>/<question id>/
```

### Phase 1 safety behaviour

The notebook records a status for every selected question:

```text
not_required
rendered
failed
disabled
```

A visual question is not considered fully supported unless at least one page image
was created successfully.

### Deliberate limitation

This phase renders the complete source page. It does not yet attempt automatic
question-only cropping. This keeps the first implementation deterministic and
prevents missing part of a figure, table or instruction.


In [ ]:

RUN_TIMESTAMP = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

VISUAL_IMAGE_ROOT = (
    OUTPUT_DIR
    / "visual_question_pages"
    / RUN_TIMESTAMP
)

VISUAL_IMAGE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


def resolve_local_pdf_path(
    raw_path: Any,
) -> Path | None:
    if raw_path is None:
        return None

    path_text = str(raw_path).strip()

    if not path_text:
        return None

    supplied_path = Path(path_text).expanduser()

    candidates = [
        supplied_path,
        PROJECT_ROOT / supplied_path,
        Path.cwd() / supplied_path,
        PROJECT_ROOT.parent / supplied_path,
    ]

    checked: set[str] = set()

    for candidate in candidates:
        try:
            resolved = candidate.resolve()
        except OSError:
            resolved = candidate

        key = str(resolved)

        if key in checked:
            continue

        checked.add(key)

        if resolved.exists() and resolved.is_file():
            return resolved

    return None


def parse_page_numbers(
    value: Any,
) -> list[int]:
    if value is None:
        return []

    if isinstance(
        value,
        (
            list,
            tuple,
            set,
            np.ndarray,
        ),
    ):
        raw_values = list(value)

    elif isinstance(value, str):
        stripped = value.strip()

        if not stripped:
            return []

        try:
            decoded = json.loads(stripped)

            if isinstance(decoded, list):
                raw_values = decoded
            else:
                raw_values = re.findall(
                    r"\d+",
                    stripped,
                )

        except json.JSONDecodeError:
            raw_values = re.findall(
                r"\d+",
                stripped,
            )

    else:
        raw_values = [value]

    page_numbers: list[int] = []

    for raw_value in raw_values:
        try:
            page_number = int(raw_value)
        except (
            TypeError,
            ValueError,
        ):
            continue

        if page_number > 0:
            page_numbers.append(page_number)

    return sorted(set(page_numbers))


def determine_question_pages(
    row: pd.Series,
) -> list[int]:
    visual_pages = parse_page_numbers(
        row.get("visual_page_numbers")
    )

    if visual_pages:
        return visual_pages

    page_start = row.get("page_start")
    page_end = row.get("page_end")

    if pd.isna(page_start):
        return []

    start = int(page_start)

    end = (
        int(page_end)
        if not pd.isna(page_end)
        else start
    )

    if end < start:
        start, end = end, start

    return list(
        range(
            start,
            end + 1,
        )
    )


def output_relative_path(
    path: Path,
) -> str:
    try:
        return path.resolve().relative_to(
            OUTPUT_DIR.resolve()
        ).as_posix()

    except ValueError:
        return path.resolve().as_posix()


def render_question_pages(
    row: pd.Series,
) -> dict[str, Any]:
    question_id = str(
        row["question_id"]
    )

    has_visual = bool(
        row["has_visual_postgres"]
    )

    base_result = {
        "question_id": question_id,
        "visual_render_required": has_visual,
        "visual_render_status": (
            "not_required"
            if not has_visual
            else "pending"
        ),
        "visual_render_error": None,
        "resolved_source_pdf_path": None,
        "source_page_numbers": [],
        "rendered_page_images": [],
        "rendered_page_count": 0,
    }

    if not has_visual:
        return base_result

    if not ENABLE_VISUAL_PAGE_RENDERING:
        base_result[
            "visual_render_status"
        ] = "disabled"

        return base_result

    pdf_path = resolve_local_pdf_path(
        row.get("source_pdf_path")
    )

    if pdf_path is None:
        base_result[
            "visual_render_status"
        ] = "failed"

        base_result[
            "visual_render_error"
        ] = (
            "The locally cached Question Paper PDF "
            "could not be resolved."
        )

        return base_result

    requested_pages = determine_question_pages(
        row
    )

    if not requested_pages:
        base_result[
            "visual_render_status"
        ] = "failed"

        base_result[
            "visual_render_error"
        ] = (
            "The question is marked as visual but "
            "contains no usable page numbers."
        )

        base_result[
            "resolved_source_pdf_path"
        ] = str(pdf_path)

        return base_result

    question_output_dir = (
        VISUAL_IMAGE_ROOT
        / question_id
    )

    question_output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    rendered_paths: list[str] = []
    valid_source_pages: list[int] = []

    try:
        with fitz.open(
            str(pdf_path)
        ) as pdf_document:
            zoom = (
                VISUAL_RENDER_DPI
                / 72.0
            )

            matrix = fitz.Matrix(
                zoom,
                zoom,
            )

            for source_page_number in (
                requested_pages
            ):
                page_index = (
                    source_page_number - 1
                    if DATABASE_PAGE_NUMBERS_ARE_ONE_BASED
                    else source_page_number
                )

                if (
                    page_index < 0
                    or page_index
                    >= pdf_document.page_count
                ):
                    continue

                page = pdf_document.load_page(
                    page_index
                )

                pixmap = page.get_pixmap(
                    matrix=matrix,
                    alpha=False,
                )

                output_file = (
                    question_output_dir
                    / (
                        f"question_{question_id}_"
                        f"page_{source_page_number:03d}.png"
                    )
                )

                pixmap.save(
                    str(output_file)
                )

                rendered_paths.append(
                    output_relative_path(
                        output_file
                    )
                )

                valid_source_pages.append(
                    source_page_number
                )

    except Exception as error:
        base_result[
            "visual_render_status"
        ] = "failed"

        base_result[
            "visual_render_error"
        ] = (
            f"{type(error).__name__}: {error}"
        )

        base_result[
            "resolved_source_pdf_path"
        ] = str(pdf_path)

        return base_result

    base_result[
        "resolved_source_pdf_path"
    ] = str(pdf_path)

    base_result[
        "source_page_numbers"
    ] = valid_source_pages

    base_result[
        "rendered_page_images"
    ] = rendered_paths

    base_result[
        "rendered_page_count"
    ] = len(rendered_paths)

    base_result[
        "visual_render_status"
    ] = (
        "rendered"
        if rendered_paths
        else "failed"
    )

    if not rendered_paths:
        base_result[
            "visual_render_error"
        ] = (
            "No requested page number was within "
            "the source PDF page range."
        )

    return base_result


visual_render_records = [
    render_question_pages(row)
    for _, row in final_df.iterrows()
]

visual_render_df = pd.DataFrame(
    visual_render_records
)

final_df = final_df.merge(
    visual_render_df,
    on="question_id",
    how="left",
    validate="one_to_one",
)


visual_manifest_path = (
    OUTPUT_DIR
    / (
        "agent2_visual_render_manifest_"
        f"{RUN_TIMESTAMP}.csv"
    )
)

visual_render_df.to_csv(
    visual_manifest_path,
    index=False,
)


selected_visual_questions = int(
    final_df[
        "has_visual_postgres"
    ].sum()
)

successfully_rendered_visual_questions = int(
    (
        final_df[
            "visual_render_status"
        ]
        == "rendered"
    ).sum()
)

visual_render_failures = int(
    (
        final_df[
            "visual_render_status"
        ]
        == "failed"
    ).sum()
)

rendered_page_total = int(
    final_df[
        "rendered_page_count"
    ].fillna(0).sum()
)


visual_render_summary_df = pd.DataFrame(
    [
        {
            "metric": (
                "selected_visual_questions"
            ),
            "value": (
                selected_visual_questions
            ),
        },
        {
            "metric": (
                "visual_questions_rendered"
            ),
            "value": (
                successfully_rendered_visual_questions
            ),
        },
        {
            "metric": (
                "visual_render_failures"
            ),
            "value": (
                visual_render_failures
            ),
        },
        {
            "metric": (
                "total_page_images_created"
            ),
            "value": rendered_page_total,
        },
    ]
)

display(visual_render_summary_df)

display(
    final_df[
        [
            "selected_rank",
            "question_id",
            "has_visual_postgres",
            "source_page_numbers",
            "visual_render_status",
            "rendered_page_count",
            "rendered_page_images",
            "visual_render_error",
        ]
    ]
)


if DISPLAY_RENDERED_IMAGES_IN_NOTEBOOK:
    for _, row in final_df.iterrows():
        image_paths = (
            row[
                "rendered_page_images"
            ]
            if isinstance(
                row[
                    "rendered_page_images"
                ],
                list,
            )
            else []
        )

        if not image_paths:
            continue

        print(
            "\n"
            f"Question {int(row['selected_rank'])} "
            "- original source page image(s)"
        )

        for relative_image_path in (
            image_paths
        ):
            absolute_image_path = (
                OUTPUT_DIR
                / relative_image_path
            )

            display(
                IPythonImage(
                    filename=str(
                        absolute_image_path
                    ),
                    width=900,
                )
            )


print(
    "Visual rendering manifest saved:"
)
print(visual_manifest_path)


## 13. Phase 3 — structured mark-scheme cleanup

### Purpose

The raw AQA marking guidance remains unchanged and remains the source of truth.

Phase 3 creates a cleaner secondary representation for the interface and evaluation:

```text
raw marking guidance              preserved
legacy structured fields          retained for comparison
Phase 3 structured fields         newly generated
```

### Deterministic classification rules

```text
Mark A / Mark B / 1 mark / Program Logic
    → marking points

A. / Accept / Allow
    → acceptable answers

R. / Reject / Do not accept
    → rejected answers

I. / Note to examiners / Maximum / Refer to
    → additional guidance

Python Example / C# Example / VB.NET Example / Correct table is
    → worked examples

AO1 / AO2 / AO3
    → assessment objectives
```

### Safety and Human-in-the-Loop behaviour

```text
raw guidance is never overwritten
low-confidence cleanup is marked review_recommended
cleaned fields are stored separately
the interface should display raw guidance prominently
Phase 3 fields are a navigational aid, not a replacement for the source
```

### Release rule

A result using lesson-summary fallback is marked `evaluation_ready` rather than
`ready_for_release`. Actual Agent 1 source chunk text must be tested before student
release.


### Phase 3 issue found during evaluation

The initial Phase 3 parser classified every extracted PDF line independently. AQA
mark-scheme instructions are often wrapped across multiple PDF lines, for example:

```text
Note to examiners: Check vertically as well as horizontally for the
effect of duplicate values.
```

The first line was classified as `additional_guidance`, while its continuation was
incorrectly placed in `marking_points`.

### Final decision

The active parser now uses logical blocks:

```text
1. clean extracted lines;
2. split inline A. / I. / R. markers;
3. detect explicit Mark / A. / R. / I. / Note / Example boundaries;
4. attach unmarked wrapped lines to the active block;
5. preserve code and table line breaks inside marking points and examples;
6. join prose continuation lines inside guidance and answer blocks;
7. export block counts, merged-line counts and block audit records.
```

Raw `marking_guidance` remains unchanged and is still the source of truth. This is
a general continuation-block approach rather than a rule for one specific question.


In [ ]:
def phase3_json_safe(value: Any) -> Any:
    if value is None:
        return None
    if isinstance(value, (bool, int, str)):
        return value
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if isinstance(value, datetime):
        return value.isoformat()
    if isinstance(value, uuid.UUID):
        return str(value)
    if isinstance(value, dict):
        return {str(k): phase3_json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [phase3_json_safe(v) for v in value]
    if hasattr(value, "item"):
        try:
            return phase3_json_safe(value.item())
        except (TypeError, ValueError):
            pass
    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass
    return str(value)


PHASE3_EXAMPLE_HEADER_PATTERN = re.compile(
    r"^(?:(?:Python|C#|VB\.NET|Java|Pseudocode)\s+)?Example\s+\d+\b.*$",
    flags=re.IGNORECASE | re.MULTILINE,
)
PHASE3_CORRECT_OUTPUT_PATTERN = re.compile(
    r"^Correct\s+(?:table|answer|output|trace)\b.*$",
    flags=re.IGNORECASE | re.MULTILINE,
)
PHASE3_ACCEPT_PATTERN = re.compile(
    r"^(?:A\.|Accept(?:able)?\b|Allow\b)", flags=re.IGNORECASE
)
PHASE3_REJECT_PATTERN = re.compile(
    r"^(?:R\.|Reject\b|Do\s+not\s+accept\b)", flags=re.IGNORECASE
)
PHASE3_GUIDANCE_PATTERN = re.compile(
    r"^(?:I\.|Note\s+to\s+examiners\b|Note\b|Maximum\b|Max\b|Refer\s+to\b)",
    flags=re.IGNORECASE,
)
PHASE3_MARKING_START_PATTERN = re.compile(
    r"^(?:\d+\s+marks?\b|Mark(?:\s+[A-Z]\b|\s+is\b)|Program\s+(?:Design|Logic)\b)",
    flags=re.IGNORECASE,
)
PHASE3_AO_PATTERN = re.compile(r"\bAO[123]\b", flags=re.IGNORECASE)
PHASE3_INLINE_MARKER_PATTERN = re.compile(r"(?<=;)\s+(?=[AIR]\.\s)", flags=re.IGNORECASE)
PHASE3_TRAILING_MARKER_PATTERN = re.compile(r"^(.*?);\s*([AIR]\.)\s*$", flags=re.IGNORECASE)
PHASE3_NOISE_LINES = {"do not", "outsid", "outside", "bo", "turn over"}


def clean_ms_line(value: Any) -> str:
    return re.sub(r"\s+", " ", str(value or "").strip())


def strip_ms_prefix(value: str) -> str:
    return re.sub(
        r"^(?:A\.|R\.|Accept(?:able)?|Allow|Reject|Do\s+not\s+accept)\s*[:\-]?\s*",
        "",
        value,
        flags=re.IGNORECASE,
    ).strip()


def expand_mark_scheme_lines(raw_text: str) -> tuple[list[str], int, list[str]]:
    expanded: list[str] = []
    split_count = 0
    dangling: list[str] = []
    pending: str | None = None

    for raw_line in raw_text.splitlines():
        line = clean_ms_line(raw_line)
        if not line:
            continue
        if pending is not None:
            line = f"{pending} {line}".strip()
            pending = None

        trailing = PHASE3_TRAILING_MARKER_PATTERN.match(line)
        if trailing:
            main = clean_ms_line(trailing.group(1) + ";")
            if main:
                expanded.append(main)
            pending = trailing.group(2).upper()
            split_count += 1
            continue

        parts = (
            [clean_ms_line(p) for p in PHASE3_INLINE_MARKER_PATTERN.split(line) if clean_ms_line(p)]
            if PHASE3_SPLIT_INLINE_MARKERS
            else [line]
        )
        split_count += max(0, len(parts) - 1)
        expanded.extend(parts)

    if pending is not None:
        dangling.append(pending)
        expanded.append(pending)

    return expanded, split_count, dangling


def classify_explicit_ms_line(line: str) -> str | None:
    if PHASE3_EXAMPLE_HEADER_PATTERN.match(line) or PHASE3_CORRECT_OUTPUT_PATTERN.match(line):
        return "worked_example"
    if PHASE3_ACCEPT_PATTERN.match(line):
        return "acceptable_answer"
    if PHASE3_REJECT_PATTERN.match(line):
        return "rejected_answer"
    if PHASE3_GUIDANCE_PATTERN.match(line):
        return "additional_guidance"
    if PHASE3_MARKING_START_PATTERN.match(line):
        return "marking_point"
    return None


def block_text(category: str, lines: list[str]) -> str:
    if category in {"additional_guidance", "acceptable_answer", "rejected_answer"}:
        return clean_ms_line(" ".join(lines))
    return "\n".join(lines).strip()


def parse_mark_scheme_guidance(raw_guidance: Any) -> dict[str, Any]:
    raw_text = str(raw_guidance or "").strip()
    empty = {
        "phase3_cleanup_status": "review_recommended",
        "phase3_rule_confidence": 0.0,
        "phase3_marking_points": [],
        "phase3_acceptable_answers": [],
        "phase3_rejected_answers": [],
        "phase3_additional_guidance": [],
        "phase3_worked_examples": [],
        "phase3_assessment_objectives": [],
        "phase3_review_reasons": ["missing_raw_marking_guidance"],
        "phase3_parser_noise_lines": [],
        "phase3_raw_guidance_preserved": True,
        "phase3_block_parser_version": PHASE3_BLOCK_PARSER_VERSION,
        "phase3_block_count": 0,
        "phase3_continuation_lines_merged": 0,
        "phase3_inline_markers_split": 0,
        "phase3_implicit_marking_block_count": 0,
        "phase3_ambiguous_lines": [],
        "phase3_dangling_inline_markers": [],
        "phase3_block_audit": [],
    }
    if not raw_text:
        return empty

    lines, inline_split, dangling = expand_mark_scheme_lines(raw_text)
    objectives = sorted({m.upper() for m in PHASE3_AO_PATTERN.findall(raw_text)})
    noise = [line for line in lines if line.lower() in PHASE3_NOISE_LINES]

    marking_points: list[str] = []
    acceptable: list[str] = []
    rejected: list[str] = []
    guidance: list[str] = []
    examples: list[dict[str, Any]] = []
    block_audit: list[dict[str, Any]] = []
    ambiguous: list[str] = []
    continuation_count = 0
    implicit_count = 0

    current_block: dict[str, Any] | None = None
    current_example: dict[str, Any] | None = None

    def flush_block() -> None:
        nonlocal current_block
        if current_block is None:
            return
        category = current_block["category"]
        lines_in_block = current_block["lines"]
        value = block_text(category, lines_in_block)
        if value:
            if category == "marking_point":
                marking_points.append(value)
            elif category == "acceptable_answer":
                cleaned = strip_ms_prefix(value)
                if cleaned:
                    acceptable.append(cleaned)
            elif category == "rejected_answer":
                cleaned = strip_ms_prefix(value)
                if cleaned:
                    rejected.append(cleaned)
            elif category == "additional_guidance":
                guidance.append(value)
            block_audit.append({
                "category": category,
                "text": value,
                "source_line_count": len(lines_in_block),
                "continuation_line_count": max(0, len(lines_in_block)-1),
                "implicit_start": bool(current_block.get("implicit_start", False)),
            })
        current_block = None

    def flush_example() -> None:
        nonlocal current_example
        if current_example is None:
            return
        content = "\n".join(current_example["lines"]).strip()
        examples.append({"header": current_example["header"], "content": content})
        block_audit.append({
            "category": "worked_example",
            "text": current_example["header"] + (("\n"+content) if content else ""),
            "source_line_count": 1 + len(current_example["lines"]),
            "continuation_line_count": len(current_example["lines"]),
            "implicit_start": False,
        })
        current_example = None

    for line in lines:
        category = classify_explicit_ms_line(line)

        if category == "worked_example":
            flush_example()
            flush_block()
            current_example = {"header": line, "lines": []}
            continue

        if current_example is not None:
            if category in {"acceptable_answer", "rejected_answer", "additional_guidance", "marking_point"}:
                flush_example()
            else:
                current_example["lines"].append(line)
                continuation_count += 1
                continue

        if category is not None:
            flush_block()
            current_block = {"category": category, "lines": [line], "implicit_start": False}
            continue

        if PHASE3_MERGE_WRAPPED_LINES and current_block is not None:
            current_block["lines"].append(line)
            continuation_count += 1
            continue

        current_block = {"category": "marking_point", "lines": [line], "implicit_start": True}
        implicit_count += 1

    flush_example()
    flush_block()

    def dedupe(values: list[Any]) -> list[Any]:
        output, seen = [], set()
        for value in values:
            key = json.dumps(value, ensure_ascii=False, sort_keys=True, default=str)
            if key not in seen:
                seen.add(key)
                output.append(value)
        return output

    marking_points = dedupe(marking_points)
    acceptable = dedupe(acceptable)
    rejected = dedupe(rejected)
    guidance = dedupe(guidance)
    examples = dedupe(examples)
    block_audit = dedupe(block_audit)

    raw_has_examples = bool(
        PHASE3_EXAMPLE_HEADER_PATTERN.search(raw_text)
        or PHASE3_CORRECT_OUTPUT_PATTERN.search(raw_text)
    )

    confidence = 0.58
    confidence += 0.08 if objectives else 0.0
    confidence += 0.08 if marking_points else 0.0
    confidence += 0.08 if (acceptable or rejected or guidance) else 0.0
    confidence += 0.10 if (not raw_has_examples or examples) else 0.0
    confidence += 0.05 if continuation_count > 0 else 0.0
    confidence += 0.03 if (inline_split > 0 or not PHASE3_SPLIT_INLINE_MARKERS) else 0.0
    confidence -= 0.20 if noise else 0.0
    confidence -= 0.12 if dangling else 0.0
    confidence -= min(0.20, 0.05*len(ambiguous))
    confidence = round(min(0.97, max(0.0, confidence)), 4)

    review: list[str] = []
    if confidence < PHASE3_MIN_RULE_CONFIDENCE:
        review.append("rule_confidence_below_threshold")
    if noise:
        review.append("parser_noise_detected_in_raw_guidance")
    if raw_has_examples and not examples:
        review.append("worked_example_header_not_segmented")
    if dangling:
        review.append("dangling_inline_marker_detected")
    if PHASE3_REVIEW_ON_AMBIGUOUS_BLOCKS and ambiguous:
        review.append("ambiguous_unassigned_mark_scheme_lines")

    return {
        "phase3_cleanup_status": "structured_ready" if not review else "review_recommended",
        "phase3_rule_confidence": confidence,
        "phase3_marking_points": marking_points,
        "phase3_acceptable_answers": acceptable,
        "phase3_rejected_answers": rejected,
        "phase3_additional_guidance": guidance,
        "phase3_worked_examples": examples,
        "phase3_assessment_objectives": objectives,
        "phase3_review_reasons": review,
        "phase3_parser_noise_lines": noise,
        "phase3_raw_guidance_preserved": True,
        "phase3_block_parser_version": PHASE3_BLOCK_PARSER_VERSION,
        "phase3_block_count": len(block_audit),
        "phase3_continuation_lines_merged": continuation_count,
        "phase3_inline_markers_split": inline_split,
        "phase3_implicit_marking_block_count": implicit_count,
        "phase3_ambiguous_lines": ambiguous,
        "phase3_dangling_inline_markers": dangling,
        "phase3_block_audit": block_audit,
    }


phase3_records = []

for _, row in final_df.iterrows():
    parsed = parse_mark_scheme_guidance(
        row[
            "marking_guidance"
        ]
    )

    legacy_payload = {
        "marking_points": (
            row["marking_points"]
        ),
        "acceptable_answers": (
            row["acceptable_answers"]
        ),
        "rejected_answers": (
            row["rejected_answers"]
        ),
        "additional_guidance": (
            row["additional_guidance"]
        ),
        "assessment_objectives": (
            row["assessment_objectives"]
        ),
    }

    cleaned_payload = {
        "marking_points": (
            parsed[
                "phase3_marking_points"
            ]
        ),
        "acceptable_answers": (
            parsed[
                "phase3_acceptable_answers"
            ]
        ),
        "rejected_answers": (
            parsed[
                "phase3_rejected_answers"
            ]
        ),
        "additional_guidance": (
            parsed[
                "phase3_additional_guidance"
            ]
        ),
        "assessment_objectives": (
            parsed[
                "phase3_assessment_objectives"
            ]
        ),
    }

    parsed[
        "question_id"
    ] = str(
        row["question_id"]
    )

    parsed[
        "mark_scheme_id"
    ] = row[
        "mark_scheme_id"
    ]

    parsed[
        "phase3_legacy_classification_changed"
    ] = bool(
        json.dumps(
            phase3_json_safe(
                legacy_payload
            ),
            sort_keys=True,
            ensure_ascii=False,
        )
        != json.dumps(
            phase3_json_safe(
                cleaned_payload
            ),
            sort_keys=True,
            ensure_ascii=False,
        )
    )

    phase3_records.append(
        parsed
    )


phase3_cleanup_df = pd.DataFrame(
    phase3_records
)

final_df = final_df.merge(
    phase3_cleanup_df,
    on=[
        "question_id",
        "mark_scheme_id",
    ],
    how="left",
    validate="one_to_one",
)


phase3_review_count = int(
    (
        final_df[
            "phase3_cleanup_status"
        ]
        == "review_recommended"
    ).sum()
)

phase3_ready_count = int(
    (
        final_df[
            "phase3_cleanup_status"
        ]
        == "structured_ready"
    ).sum()
)

phase3_changes_count = int(
    final_df[
        "phase3_legacy_classification_changed"
    ].sum()
)

phase3_mean_confidence = float(
    final_df[
        "phase3_rule_confidence"
    ].mean()
)


all_topics_use_actual_chunk_evidence = bool(
    fallback_evidence_topic_count
    == 0
)


release_blockers = []

if not selection_summary[
    "marks_within_tolerance"
]:
    release_blockers.append(
        "target_marks_outside_tolerance"
    )

if selection_summary[
    "semantic_rescue_selected"
]:
    release_blockers.append(
        "semantic_rescue_selected"
    )

if (
    REQUIRE_ACTUAL_AGENT1_CHUNK_EVIDENCE_FOR_RELEASE
    and not all_topics_use_actual_chunk_evidence
):
    release_blockers.append(
        "actual_agent1_chunk_evidence_not_used"
    )

if phase3_review_count > 0:
    release_blockers.append(
        "phase3_mark_scheme_review_recommended"
    )


hard_decision_blockers = {
    "target_marks_outside_tolerance",
    "semantic_rescue_selected",
}

if hard_decision_blockers.intersection(
    release_blockers
):
    final_release_status = (
        "needs_user_decision"
    )

elif release_blockers:
    final_release_status = (
        "evaluation_ready"
    )

else:
    final_release_status = (
        "ready_for_release"
    )


selection_summary[
    "assessment_release_status"
] = final_release_status

selection_summary[
    "release_blockers"
] = release_blockers

selection_summary[
    "requires_user_decision"
] = bool(
    final_release_status
    == "needs_user_decision"
)

selection_summary[
    "all_topics_use_actual_chunk_evidence"
] = (
    all_topics_use_actual_chunk_evidence
)

selection_summary[
    "phase3_review_count"
] = phase3_review_count


phase3_summary = {
    "phase3_version": PHASE3_VERSION,
    "phase3_enabled": (
        ENABLE_PHASE3_MARK_SCHEME_CLEANUP
    ),
    "raw_guidance_source_of_truth": (
        PHASE3_RAW_GUIDANCE_IS_SOURCE_OF_TRUTH
    ),
    "selected_mark_schemes": int(
        len(final_df)
    ),
    "structured_ready_count": (
        phase3_ready_count
    ),
    "review_recommended_count": (
        phase3_review_count
    ),
    "legacy_classification_changed_count": (
        phase3_changes_count
    ),
    "mean_rule_confidence": round(
        phase3_mean_confidence,
        4,
    ),
    "block_parser_version": PHASE3_BLOCK_PARSER_VERSION,
    "total_blocks_created": int(final_df["phase3_block_count"].sum()),
    "total_continuation_lines_merged": int(
        final_df["phase3_continuation_lines_merged"].sum()
    ),
    "total_inline_markers_split": int(
        final_df["phase3_inline_markers_split"].sum()
    ),
    "total_ambiguous_lines": int(
        final_df["phase3_ambiguous_lines"].map(len).sum()
    ),
    "actual_agent1_chunk_evidence_used": (
        all_topics_use_actual_chunk_evidence
    ),
    "final_release_status": (
        final_release_status
    ),
    "release_blockers": (
        release_blockers
    ),
}


phase3_manifest_path = (
    OUTPUT_DIR
    / (
        "agent2_phase3_mark_scheme_cleanup_"
        f"{RUN_TIMESTAMP}.csv"
    )
)

phase3_json_path = (
    OUTPUT_DIR
    / (
        "agent2_phase3_mark_scheme_cleanup_"
        f"{RUN_TIMESTAMP}.json"
    )
)


phase3_export_df = (
    phase3_cleanup_df.copy()
)

for column in [
    "phase3_marking_points",
    "phase3_acceptable_answers",
    "phase3_rejected_answers",
    "phase3_additional_guidance",
    "phase3_worked_examples",
    "phase3_assessment_objectives",
    "phase3_review_reasons",
    "phase3_parser_noise_lines",
    "phase3_ambiguous_lines",
    "phase3_dangling_inline_markers",
    "phase3_block_audit",
]:
    phase3_export_df[
        column
    ] = phase3_export_df[
        column
    ].map(
        lambda value: json.dumps(
            phase3_json_safe(value),
            ensure_ascii=False,
        )
    )

phase3_export_df.to_csv(
    phase3_manifest_path,
    index=False,
)

phase3_json_path.write_text(
    json.dumps(
        phase3_json_safe(
            {
                "summary": (
                    phase3_summary
                ),
                "records": (
                    phase3_cleanup_df
                    .to_dict(
                        orient="records"
                    )
                ),
            }
        ),
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


display(
    pd.DataFrame(
        [phase3_summary]
    )
)

display(
    final_df[
        [
            "selected_rank",
            "question_id",
            "mark_scheme_id",
            "phase3_cleanup_status",
            "phase3_rule_confidence",
            "phase3_block_parser_version",
            "phase3_block_count",
            "phase3_continuation_lines_merged",
            "phase3_inline_markers_split",
            "phase3_implicit_marking_block_count",
            "phase3_ambiguous_lines",
            "phase3_legacy_classification_changed",
            "phase3_review_reasons",
            "phase3_marking_points",
            "phase3_acceptable_answers",
            "phase3_rejected_answers",
            "phase3_additional_guidance",
            "phase3_worked_examples",
            "phase3_assessment_objectives",
        ]
    ]
)

print(
    "Final assessment release status: "
    f"{final_release_status}"
)

print(
    "Release blockers: "
    f"{release_blockers}"
)

print(
    "Phase 3 manifest saved:"
)
print(phase3_manifest_path)
print(phase3_json_path)


In [ ]:
# Phase 3 block-aware parser validation
wrapped_guidance_test = parse_mark_scheme_guidance(
    """
    Mark is for AO2 (apply)
    The first iteration structure processes the rows;
    Note to examiners: award both marks if the student
    answers are correct but the opposite way around,
    and rows are given for the second answer
    """
)
expected_guidance = (
    "Note to examiners: award both marks if the student "
    "answers are correct but the opposite way around, "
    "and rows are given for the second answer"
)

inline_marker_test = parse_mark_scheme_guidance(
    """
    Mark F for resetting pos to 0; A. if the index could go out of range.
    Mark D for checking the input; I.
    data validation attempts
    """
)

phase3_block_parser_validation = {
    "wrapped_guidance_merged_correctly": (
        expected_guidance in wrapped_guidance_test["phase3_additional_guidance"]
    ),
    "wrapped_guidance_not_in_marking_points": all(
        "answers are correct but the opposite way around" not in value
        for value in wrapped_guidance_test["phase3_marking_points"]
    ),
    "inline_accept_split_correctly": any(
        "if the index could go out of range" in value.lower()
        for value in inline_marker_test["phase3_acceptable_answers"]
    ),
    "trailing_guidance_marker_merged": any(
        "data validation attempts" in value.lower()
        for value in inline_marker_test["phase3_additional_guidance"]
    ),
}
display(pd.DataFrame([
    {"check": key, "passed": bool(value)}
    for key, value in phase3_block_parser_validation.items()
]))
if not all(phase3_block_parser_validation.values()):
    raise RuntimeError("Phase 3 block-aware parser validation failed.")
print("Phase 3 block-aware parser validation: passed")

## 14. Build and export the final assessment package

Outputs:

```text
retrieval candidates CSV
selected questions CSV
complete assessment JSON
readable assessment Markdown
plain-text evaluation report
```


In [ ]:
def json_safe(value: Any) -> Any:
    if value is None:
        return None

    if isinstance(
        value,
        (
            bool,
            int,
            str,
        ),
    ):
        return value

    if isinstance(value, float):
        return (
            value
            if math.isfinite(value)
            else None
        )

    if isinstance(value, datetime):
        return value.isoformat()

    if isinstance(value, uuid.UUID):
        return str(value)

    if isinstance(value, dict):
        return {
            str(key): json_safe(item)
            for key, item
            in value.items()
        }

    if isinstance(
        value,
        (
            list,
            tuple,
            set,
        ),
    ):
        return [
            json_safe(item)
            for item in value
        ]

    if hasattr(value, "item"):
        try:
            return json_safe(
                value.item()
            )
        except (
            TypeError,
            ValueError,
        ):
            pass

    try:
        if pd.isna(value):
            return None
    except (
        TypeError,
        ValueError,
    ):
        pass

    return str(value)


question_packages = []

for _, row in final_df.iterrows():
    question_packages.append(
        {
            "rank": int(
                row["selected_rank"]
            ),
            "question_id": (
                row["question_id"]
            ),
            "topic": {
                "detected_topic": (
                    row["detected_topic"]
                ),
                "role": (
                    row["agent1_role"]
                ),
                "official_reference": (
                    row[
                        "official_reference_postgres"
                    ]
                ),
                "official_concept_name": (
                    row[
                        "official_concept_name_postgres"
                    ]
                ),
                "section_reference": (
                    row[
                        "official_section_reference_postgres"
                    ]
                ),
                "pmt_subtopic_code": (
                    row["pmt_subtopic_code"]
                ),
                "pmt_subtopic_name": (
                    row["pmt_subtopic_name"]
                ),
            },
            "question": {
                "number": (
                    row[
                        "question_number_postgres"
                    ]
                ),
                "text": (
                    row[
                        "question_text_postgres"
                    ]
                ),
                "context": (
                    row["context_text"]
                ),
                "marks": int(
                    row["marks_postgres"]
                ),
                "has_code": bool(
                    row["has_code_postgres"]
                ),
                "has_visual": bool(
                    row["has_visual_postgres"]
                ),
                "paper_code": (
                    row["paper_code_postgres"]
                ),
                "programming_language": (
                    row["programming_language_postgres"]
                ),
                "source_pdf_path": (
                    row[
                        "resolved_source_pdf_path"
                    ]
                ),
                "source_page_numbers": (
                    row[
                        "source_page_numbers"
                    ]
                ),
                "rendered_page_images": (
                    row[
                        "rendered_page_images"
                    ]
                ),
                "visual_render_status": (
                    row[
                        "visual_render_status"
                    ]
                ),
                "visual_render_error": (
                    row[
                        "visual_render_error"
                    ]
                ),
            },
            "mark_scheme": {
                "id": str(
                    row["mark_scheme_id"]
                ),
                "maximum_marks": int(
                    row["maximum_marks"]
                ),
                "marking_guidance": (
                    row[
                        "marking_guidance"
                    ]
                ),
                "raw_marking_guidance": (
                    row[
                        "marking_guidance"
                    ]
                ),
                "legacy_structured_fields": {
                    "marking_points": (
                        row[
                            "marking_points"
                        ]
                    ),
                    "acceptable_answers": (
                        row[
                            "acceptable_answers"
                        ]
                    ),
                    "rejected_answers": (
                        row[
                            "rejected_answers"
                        ]
                    ),
                    "additional_guidance": (
                        row[
                            "additional_guidance"
                        ]
                    ),
                    "assessment_objectives": (
                        row[
                            "assessment_objectives"
                        ]
                    ),
                },
                "phase3_structured": {
                    "version": (
                        PHASE3_VERSION
                    ),
                    "cleanup_status": (
                        row[
                            "phase3_cleanup_status"
                        ]
                    ),
                    "rule_confidence": float(
                        row[
                            "phase3_rule_confidence"
                        ]
                    ),
                    "marking_points": (
                        row[
                            "phase3_marking_points"
                        ]
                    ),
                    "acceptable_answers": (
                        row[
                            "phase3_acceptable_answers"
                        ]
                    ),
                    "rejected_answers": (
                        row[
                            "phase3_rejected_answers"
                        ]
                    ),
                    "additional_guidance": (
                        row[
                            "phase3_additional_guidance"
                        ]
                    ),
                    "worked_examples": (
                        row[
                            "phase3_worked_examples"
                        ]
                    ),
                    "assessment_objectives": (
                        row[
                            "phase3_assessment_objectives"
                        ]
                    ),
                    "review_reasons": (
                        row[
                            "phase3_review_reasons"
                        ]
                    ),
                    "block_parser_version": row["phase3_block_parser_version"],
                    "block_count": int(row["phase3_block_count"]),
                    "continuation_lines_merged": int(
                        row["phase3_continuation_lines_merged"]
                    ),
                    "inline_markers_split": int(
                        row["phase3_inline_markers_split"]
                    ),
                    "implicit_marking_block_count": int(
                        row["phase3_implicit_marking_block_count"]
                    ),
                    "ambiguous_lines": row["phase3_ambiguous_lines"],
                    "block_audit": row["phase3_block_audit"],
                    "raw_guidance_preserved": bool(
                        row[
                            "phase3_raw_guidance_preserved"
                        ]
                    ),
                    "legacy_classification_changed": bool(
                        row[
                            "phase3_legacy_classification_changed"
                        ]
                    ),
                },
            },
            "retrieval": {
                "stage": (
                    row["retrieval_stage"]
                ),
                "semantic_score": float(
                    row["semantic_score"]
                ),
                "final_score": float(
                    row["final_score"]
                ),
                "agent1_confidence": float(
                    row[
                        "agent1_confidence"
                    ]
                ),
                "source_chunks": (
                    row["source_chunks"]
                ),
                "query_evidence_source": (
                    row["query_evidence_source"]
                ),
                "phase2_version": PHASE2_VERSION,
                "phase2_semantic_threshold": (
                    row["phase2_semantic_threshold"]
                ),
                "concept_gate_passed": bool(
                    row["concept_gate_passed"]
                ),
                "question_quality_gate_passed": bool(
                    row[
                        "question_quality_gate_passed"
                    ]
                ),
                "phase2_gate_passed": bool(
                    row["phase2_gate_passed"]
                ),
                "phase2_gate_mode": (
                    row["phase2_gate_mode"]
                ),
                "semantic_rescue_used": bool(
                    row[
                        "semantic_rescue_used"
                    ]
                ),
                "question_quality_issues": (
                    row["question_quality_issues"]
                ),
                "phase2_rejection_reasons": (
                    row["phase2_rejection_reasons"]
                ),
                "qp_ms_match_method": (
                    row["match_method"]
                ),
                "qp_ms_match_confidence": (
                    float(
                        row[
                            "match_confidence"
                        ]
                    )
                ),
            },
        }
    )


retrieval_summary = {
    "visual_rendering_version": (
        VISUAL_RENDERING_VERSION
    ),
    "selected_visual_questions": (
        selected_visual_questions
    ),
    "visual_questions_rendered": (
        successfully_rendered_visual_questions
    ),
    "visual_render_failures": (
        visual_render_failures
    ),
    "rendered_page_images": (
        rendered_page_total
    ),
    **phase2_gate_summary,
    "raw_candidates": len(
        all_candidates_df
    ),
    "duplicates_removed": (
        duplicates_removed
    ),
    "near_duplicates_removed": (
        near_duplicates_removed
    ),
    "unique_candidates": len(
        unique_candidates_df
    ),
    "refined_unique_candidates": len(
        near_unique_candidates_df
    ),
    **selection_summary,
    **phase3_summary,
}


assessment_package = json_safe(
    {
        "generated_at_utc": (
            datetime.now(
                timezone.utc
            ).isoformat()
        ),
        "retrieval_version": (
            RETRIEVAL_VERSION
        ),
        "specification": {
            "code": SPECIFICATION_CODE,
            "version": (
                SPECIFICATION_VERSION
            ),
        },
        "embedding_model": MODEL_NAME,
        "qdrant_collection": (
            AGENT2_COLLECTION
        ),
        "phase_1_visual_rendering": {
            "version": (
                VISUAL_RENDERING_VERSION
            ),
            "strategy": (
                "original_full_question_pages"
            ),
            "enabled": (
                ENABLE_VISUAL_PAGE_RENDERING
            ),
            "render_dpi": (
                VISUAL_RENDER_DPI
            ),
            "manifest": (
                visual_manifest_path
                .relative_to(
                    OUTPUT_DIR
                )
                .as_posix()
            ),
            "known_limitation": (
                "Full source pages are rendered; "
                "automatic question-only cropping "
                "is a future refinement."
            ),
        },
        "phase_2_concept_quality_gate": {
            "version": PHASE2_VERSION,
            "refinement_version": (
                PHASE2_REFINEMENT_VERSION
            ),
            "near_duplicate_gate": {
                "enabled": (
                    ENABLE_NEAR_DUPLICATE_GATE
                ),
                "lexical_threshold": (
                    NEAR_DUPLICATE_LEXICAL_THRESHOLD
                ),
                "token_jaccard_threshold": (
                    NEAR_DUPLICATE_TOKEN_JACCARD_THRESHOLD
                ),
                "semantic_threshold": (
                    NEAR_DUPLICATE_SEMANTIC_THRESHOLD
                ),
                "near_duplicates_removed": (
                    near_duplicates_removed
                ),
            },
            "threshold_strategy": (
                "per_topic_adaptive"
            ),
            "adaptive_threshold_version": (
                ADAPTIVE_THRESHOLD_VERSION
            ),
            "legacy_fixed_threshold_history": {
                "strict_threshold": (
                    LEGACY_FIXED_STRICT_THRESHOLD
                ),
                "relaxed_threshold": (
                    LEGACY_FIXED_RELAXED_THRESHOLD
                ),
                "active": (
                    LEGACY_FIXED_THRESHOLD_APPROACH_ENABLED
                ),
            },
            "adaptive_configuration": {
                "score_percentile": (
                    ADAPTIVE_SCORE_PERCENTILE
                ),
                "top_score_margin": (
                    ADAPTIVE_TOP_SCORE_MARGIN
                ),
                "absolute_minimum_score": (
                    ADAPTIVE_ABSOLUTE_MINIMUM_SCORE
                ),
                "maximum_allowed_threshold": (
                    ADAPTIVE_MAXIMUM_ALLOWED_THRESHOLD
                ),
            },
            "adaptive_threshold_summary": {
                "minimum": (
                    adaptive_threshold_min
                ),
                "maximum": (
                    adaptive_threshold_max
                ),
                "mean": (
                    adaptive_threshold_mean
                ),
                "pool_sufficient_without_rescue": (
                    adaptive_pool_sufficient
                ),
            },
            "quality_safe_rescue": {
                "enabled": (
                    ENABLE_QUALITY_SAFE_TOPIC_RESCUE
                ),
                "used": (
                    phase2_rescue_used
                ),
                "minimum_semantic_score": (
                    MIN_TOPIC_RESCUE_SEMANTIC_SCORE
                ),
                "selected_rescue_candidate": (
                    selection_summary[
                        "semantic_rescue_selected"
                    ]
                ),
            },
            "quality_gate_enabled": (
                ENABLE_QUESTION_TEXT_QUALITY_GATE
            ),
            "minimum_question_word_count": (
                MIN_QUESTION_WORD_COUNT
            ),
            "known_limitations": [
                (
                    "Threshold remains provisional until "
                    "Notebook 06 evaluation."
                ),
                (
                    "Lesson summary is used when actual "
                    "Agent 1 chunk text is unavailable."
                ),
            ],
        },
        "phase_3_mark_scheme_cleanup": {
            "version": (
                PHASE3_VERSION
            ),
            "enabled": (
                ENABLE_PHASE3_MARK_SCHEME_CLEANUP
            ),
            "raw_guidance_source_of_truth": (
                PHASE3_RAW_GUIDANCE_IS_SOURCE_OF_TRUTH
            ),
            "minimum_rule_confidence": (
                PHASE3_MIN_RULE_CONFIDENCE
            ),
            "manifest": (
                phase3_manifest_path
                .relative_to(
                    OUTPUT_DIR
                )
                .as_posix()
            ),
            "json_report": (
                phase3_json_path
                .relative_to(
                    OUTPUT_DIR
                )
                .as_posix()
            ),
            "summary": (
                phase3_summary
            ),
        },
        "lesson_summary": (
            LESSON_SUMMARY
        ),
        "assessment_request": request,
        "agent1_topics": (
            validated_topics_df[
                [
                    "detected_topic",
                    "role",
                    "official_reference",
                    "confidence",
                    "ranking_score",
                    "source_chunks",
                    "query_evidence_source",
                    "query_evidence_item_count",
                ]
            ]
            .to_dict(
                orient="records"
            )
        ),
        "retrieval_summary": (
            retrieval_summary
        ),
        "questions": question_packages,
    }
)


timestamp = RUN_TIMESTAMP

candidates_path = (
    OUTPUT_DIR
    / f"agent2_retrieval_candidates_{timestamp}.csv"
)

selected_path = (
    OUTPUT_DIR
    / f"agent2_selected_questions_{timestamp}.csv"
)

package_path = (
    OUTPUT_DIR
    / f"agent2_assessment_package_{timestamp}.json"
)

markdown_path = (
    OUTPUT_DIR
    / f"agent2_assessment_package_{timestamp}.md"
)

text_report_path = (
    OUTPUT_DIR
    / f"agent2_assessment_evaluation_{timestamp}.txt"
)

phase2_gate_manifest_path = (
    OUTPUT_DIR
    / f"agent2_phase2_gate_manifest_{timestamp}.csv"
)

phase2_query_evidence_path = (
    OUTPUT_DIR
    / f"agent2_phase2_query_evidence_{timestamp}.csv"
)

near_duplicate_manifest_path = (
    OUTPUT_DIR
    / f"agent2_phase2_near_duplicate_manifest_{timestamp}.csv"
)

adaptive_threshold_profile_path = (
    OUTPUT_DIR
    / f"agent2_phase2_adaptive_threshold_profile_{timestamp}.csv"
)

release_readiness_path = (
    OUTPUT_DIR
    / f"agent2_assessment_release_readiness_{timestamp}.json"
)


phase2_gate_manifest_df.to_csv(
    candidates_path,
    index=False,
)

phase2_gate_manifest_df.to_csv(
    phase2_gate_manifest_path,
    index=False,
)

phase2_query_evidence_df.to_csv(
    phase2_query_evidence_path,
    index=False,
)

near_duplicate_manifest_df.to_csv(
    near_duplicate_manifest_path,
    index=False,
)

adaptive_threshold_profile_df.to_csv(
    adaptive_threshold_profile_path,
    index=False,
)

release_readiness_payload = {
    "assessment_release_status": (
        selection_summary[
            "assessment_release_status"
        ]
    ),
    "requires_user_decision": (
        selection_summary[
            "requires_user_decision"
        ]
    ),
    "target_marks": (
        selection_summary[
            "target_marks"
        ]
    ),
    "selected_marks": (
        selection_summary[
            "selected_marks"
        ]
    ),
    "marks_difference": (
        selection_summary[
            "marks_difference"
        ]
    ),
    "target_marks_tolerance": (
        TARGET_MARKS_TOLERANCE
    ),
    "semantic_rescue_used_in_pool": (
        phase2_rescue_used
    ),
    "semantic_rescue_selected": (
        selection_summary[
            "semantic_rescue_selected"
        ]
    ),
    "threshold_strategy": (
        "per_topic_adaptive"
    ),
    "adaptive_threshold_version": (
        ADAPTIVE_THRESHOLD_VERSION
    ),
    "adaptive_threshold_min": (
        adaptive_threshold_min
    ),
    "adaptive_threshold_max": (
        adaptive_threshold_max
    ),
    "adaptive_threshold_mean": (
        adaptive_threshold_mean
    ),
    "rescue_semantic_floor": (
        MIN_TOPIC_RESCUE_SEMANTIC_SCORE
    ),
    "required_official_references": (
        selection_summary[
            "required_official_references"
        ]
    ),
    "selected_official_references": (
        selection_summary[
            "selected_official_references"
        ]
    ),
    "all_required_references_covered": (
        selection_summary[
            "all_required_references_covered"
        ]
    ),
    "all_topics_use_actual_chunk_evidence": (
        selection_summary[
            "all_topics_use_actual_chunk_evidence"
        ]
    ),
    "phase3_review_count": (
        selection_summary[
            "phase3_review_count"
        ]
    ),
    "release_blockers": (
        selection_summary[
            "release_blockers"
        ]
    ),
}

release_readiness_path.write_text(
    json.dumps(
        json_safe(
            release_readiness_payload
        ),
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

final_df.to_csv(
    selected_path,
    index=False,
)

package_path.write_text(
    json.dumps(
        assessment_package,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


markdown_lines = [
    "# Generated Assessment",
    "",
    (
        f"**Total marks:** "
        f"{selection_summary['selected_marks']}"
    ),
    "",
]

for question in question_packages:
    markdown_lines.extend(
        [
            (
                f"## Question {question['rank']} "
                f"({question['question']['marks']} marks)"
            ),
            "",
            (
                f"**Topic:** "
                f"{question['topic']['detected_topic']} "
                f"({question['topic']['official_reference']})"
            ),
            "",
            question["question"]["text"],
            "",
        ]
    )

    context_text = str(
        question["question"].get(
            "context"
        )
        or ""
    ).strip()

    if context_text:
        markdown_lines.extend(
            [
                "**Context**",
                "",
                context_text,
                "",
            ]
        )

    rendered_images = (
        question["question"].get(
            "rendered_page_images"
        )
        or []
    )

    if rendered_images:
        markdown_lines.extend(
            [
                "### Original question page image(s)",
                "",
                (
                    "The following page image(s) were rendered "
                    "from the original cached Question Paper PDF."
                ),
                "",
            ]
        )

        for rendered_image in rendered_images:
            markdown_lines.extend(
                [
                    (
                        f"![Original question page]"
                        f"({rendered_image})"
                    ),
                    "",
                ]
            )

    phase3_ms = (
        question[
            "mark_scheme"
        ][
            "phase3_structured"
        ]
    )

    markdown_lines.extend(
        [
            "### Mark scheme — raw guidance",
            "",
            str(
                question[
                    "mark_scheme"
                ][
                    "raw_marking_guidance"
                ]
                or ""
            ),
            "",
            "### Phase 3 structured view",
            "",
            (
                f"**Cleanup status:** "
                f"{phase3_ms['cleanup_status']}"
            ),
            "",
            (
                f"**Rule confidence:** "
                f"{phase3_ms['rule_confidence']}"
            ),
            "",
            f"**Block parser:** {phase3_ms['block_parser_version']}",
            "",
            f"**Wrapped lines merged:** {phase3_ms['continuation_lines_merged']}",
            "",
            f"**Inline markers split:** {phase3_ms['inline_markers_split']}",
            "",
            "**Marking points**",
            "",
            json.dumps(
                phase3_ms[
                    "marking_points"
                ],
                indent=2,
                ensure_ascii=False,
            ),
            "",
            "**Acceptable answers**",
            "",
            json.dumps(
                phase3_ms[
                    "acceptable_answers"
                ],
                indent=2,
                ensure_ascii=False,
            ),
            "",
            "**Rejected answers**",
            "",
            json.dumps(
                phase3_ms[
                    "rejected_answers"
                ],
                indent=2,
                ensure_ascii=False,
            ),
            "",
            "**Additional guidance**",
            "",
            json.dumps(
                phase3_ms[
                    "additional_guidance"
                ],
                indent=2,
                ensure_ascii=False,
            ),
            "",
            "**Worked examples**",
            "",
            json.dumps(
                phase3_ms[
                    "worked_examples"
                ],
                indent=2,
                ensure_ascii=False,
            ),
            "",
            "---",
            "",
        ]
    )


markdown_path.write_text(
    "\n".join(markdown_lines),
    encoding="utf-8",
)


# ---------------------------------------------------------
# Plain-text evaluation report
# ---------------------------------------------------------
text_lines = [
    "=" * 78,
    "AGENT 2 — NOTEBOOK 05 RETRIEVAL EVALUATION REPORT",
    "=" * 78,
    "",
    f"Generated at (UTC): {assessment_package['generated_at_utc']}",
    f"Specification: {SPECIFICATION_CODE}",
    f"Specification version: {SPECIFICATION_VERSION}",
    f"Retrieval version: {RETRIEVAL_VERSION}",
    f"Embedding model: {MODEL_NAME}",
    f"Qdrant collection: {AGENT2_COLLECTION}",
    "",
    "-" * 78,
    "AGENT 1 DETECTED TOPICS",
    "-" * 78,
]

for topic_index, topic in enumerate(
    assessment_package["agent1_topics"],
    start=1,
):
    text_lines.extend(
        [
            f"Topic {topic_index}",
            f"  Name: {topic['detected_topic']}",
            f"  Role: {topic['role']}",
            f"  Official reference: {topic['official_reference']}",
            f"  Confidence: {topic['confidence']}",
            f"  Ranking score: {topic['ranking_score']}",
            f"  Source chunks: {topic['source_chunks']}",
            "",
        ]
    )

text_lines.extend(
    [
        "-" * 78,
        "ASSESSMENT REQUEST",
        "-" * 78,
    ]
)

for key, value in request.items():
    text_lines.append(f"{key}: {value}")

text_lines.extend(
    [
        "",
        f"Lesson summary: {LESSON_SUMMARY}",
        "",
        "-" * 78,
        "RETRIEVAL SUMMARY",
        "-" * 78,
    ]
)

for key, value in retrieval_summary.items():
    text_lines.append(f"{key}: {value}")

text_lines.extend(
    [
        "",
        "-" * 78,
        "SELECTED QUESTIONS AND MARK SCHEMES",
        "-" * 78,
        "",
    ]
)

for question in question_packages:
    text_lines.extend(
        [
            "=" * 78,
            (
                f"QUESTION {question['rank']} "
                f"({question['question']['marks']} marks)"
            ),
            "=" * 78,
            f"Question ID: {question['question_id']}",
            f"Detected topic: {question['topic']['detected_topic']}",
            f"Topic role: {question['topic']['role']}",
            (
                "Official reference: "
                f"{question['topic']['official_reference']}"
            ),
            (
                "Official concept: "
                f"{question['topic']['official_concept_name']}"
            ),
            (
                "PMT source topic: "
                f"{question['topic']['pmt_subtopic_code']} — "
                f"{question['topic']['pmt_subtopic_name']}"
            ),
            f"Paper code: {question['question']['paper_code']}",
            (
                "Programming language: "
                f"{question['question']['programming_language']}"
            ),
            (
                "Contains code: "
                f"{question['question']['has_code']}"
            ),
            (
                "Contains visual: "
                f"{question['question']['has_visual']}"
            ),
            (
                "Visual render status: "
                f"{question['question']['visual_render_status']}"
            ),
            (
                "Original source pages: "
                f"{question['question']['source_page_numbers']}"
            ),
            (
                "Rendered page images: "
                f"{question['question']['rendered_page_images']}"
            ),
            (
                "Visual rendering error: "
                f"{question['question']['visual_render_error']}"
            ),
            "",
            "QUESTION TEXT",
            "-" * 78,
            str(question["question"]["text"] or ""),
            "",
        ]
    )

    context_text = str(
        question["question"].get("context") or ""
    ).strip()

    if context_text:
        text_lines.extend(
            [
                "CONTEXT",
                "-" * 78,
                context_text,
                "",
            ]
        )

    mark_scheme = question["mark_scheme"]

    text_lines.extend(
        [
            "MARK SCHEME",
            "-" * 78,
            (
                "Maximum marks: "
                f"{mark_scheme['maximum_marks']}"
            ),
            "",
            "Marking guidance:",
            str(mark_scheme["marking_guidance"] or ""),
            "",
            "PHASE 3 STRUCTURED MARK SCHEME",
            "-" * 78,
            (
                "Cleanup status: "
                f"{mark_scheme['phase3_structured']['cleanup_status']}"
            ),
            (
                "Rule confidence: "
                f"{mark_scheme['phase3_structured']['rule_confidence']}"
            ),
            (
                "Review reasons: "
                f"{mark_scheme['phase3_structured']['review_reasons']}"
            ),
            (
                "Block parser version: "
                f"{mark_scheme['phase3_structured']['block_parser_version']}"
            ),
            (
                "Logical blocks created: "
                f"{mark_scheme['phase3_structured']['block_count']}"
            ),
            (
                "Wrapped continuation lines merged: "
                f"{mark_scheme['phase3_structured']['continuation_lines_merged']}"
            ),
            (
                "Inline markers split: "
                f"{mark_scheme['phase3_structured']['inline_markers_split']}"
            ),
            (
                "Ambiguous lines: "
                f"{mark_scheme['phase3_structured']['ambiguous_lines']}"
            ),
            "",
            "Marking points:",
            json.dumps(
                mark_scheme[
                    "phase3_structured"
                ][
                    "marking_points"
                ],
                indent=2,
                ensure_ascii=False,
            ),
            "",
            "Acceptable answers:",
            json.dumps(
                mark_scheme[
                    "phase3_structured"
                ][
                    "acceptable_answers"
                ],
                indent=2,
                ensure_ascii=False,
            ),
            "",
            "Rejected answers:",
            json.dumps(
                mark_scheme[
                    "phase3_structured"
                ][
                    "rejected_answers"
                ],
                indent=2,
                ensure_ascii=False,
            ),
            "",
            "Additional guidance:",
            json.dumps(
                mark_scheme[
                    "phase3_structured"
                ][
                    "additional_guidance"
                ],
                indent=2,
                ensure_ascii=False,
            ),
            "",
            "Worked examples:",
            json.dumps(
                mark_scheme[
                    "phase3_structured"
                ][
                    "worked_examples"
                ],
                indent=2,
                ensure_ascii=False,
            ),
            "",
            "Assessment objectives:",
            json.dumps(
                mark_scheme[
                    "phase3_structured"
                ][
                    "assessment_objectives"
                ],
                indent=2,
                ensure_ascii=False,
            ),
            "",
            (
                "Legacy structured fields retained "
                "in JSON/CSV for comparison."
            ),
            "",
            "RETRIEVAL EVIDENCE",
            "-" * 78,
            (
                "Retrieval stage: "
                f"{question['retrieval']['stage']}"
            ),
            (
                "Semantic score: "
                f"{question['retrieval']['semantic_score']:.6f}"
            ),
            (
                "Final score: "
                f"{question['retrieval']['final_score']:.6f}"
            ),
            (
                "Agent 1 confidence: "
                f"{question['retrieval']['agent1_confidence']:.6f}"
            ),
            (
                "Source chunks: "
                f"{question['retrieval']['source_chunks']}"
            ),
            (
                "Query evidence source: "
                f"{question['retrieval']['query_evidence_source']}"
            ),
            (
                "Phase 2 threshold: "
                f"{question['retrieval']['phase2_semantic_threshold']}"
            ),
            (
                "Concept gate passed: "
                f"{question['retrieval']['concept_gate_passed']}"
            ),
            (
                "Question-quality gate passed: "
                f"{question['retrieval']['question_quality_gate_passed']}"
            ),
            (
                "Phase 2 gate mode: "
                f"{question['retrieval']['phase2_gate_mode']}"
            ),
            (
                "Semantic rescue used: "
                f"{question['retrieval']['semantic_rescue_used']}"
            ),
            (
                "Question-quality issues: "
                f"{question['retrieval']['question_quality_issues']}"
            ),
            (
                "QP/MS match method: "
                f"{question['retrieval']['qp_ms_match_method']}"
            ),
            (
                "QP/MS match confidence: "
                f"{question['retrieval']['qp_ms_match_confidence']:.6f}"
            ),
            "",
        ]
    )

text_lines.extend(
    [
        "=" * 78,
        "END OF REPORT",
        "=" * 78,
    ]
)

text_report_path.write_text(
    "\n".join(text_lines),
    encoding="utf-8",
)


print("Saved:")
print(candidates_path)
print(selected_path)
print(package_path)
print(markdown_path)
print(text_report_path)
print(visual_manifest_path)
print(VISUAL_IMAGE_ROOT)
print(phase2_gate_manifest_path)
print(phase2_query_evidence_path)
print(near_duplicate_manifest_path)
print(adaptive_threshold_profile_path)
print(release_readiness_path)
print(phase3_manifest_path)
print(phase3_json_path)


## 15. Store retrieval audit logs

The audit tables record the Agent 1 input, request, candidates, scores and final
selections. These records will support Notebook 06 retrieval evaluation.


In [ ]:

AUDIT_SQL = [
    """
    CREATE TABLE IF NOT EXISTS assessment_retrieval_runs
    (
        id UUID PRIMARY KEY,
        retrieval_version VARCHAR(120) NOT NULL,
        specification_code VARCHAR(20) NOT NULL,
        specification_version VARCHAR(120) NOT NULL,
        embedding_model VARCHAR(200) NOT NULL,
        qdrant_collection VARCHAR(200) NOT NULL,
        agent1_input JSONB NOT NULL,
        assessment_request JSONB NOT NULL,
        lesson_summary TEXT,
        raw_candidate_count INTEGER NOT NULL,
        duplicate_count INTEGER NOT NULL,
        unique_candidate_count INTEGER NOT NULL,
        selected_question_count INTEGER NOT NULL,
        selected_total_marks INTEGER NOT NULL,
        status VARCHAR(40) NOT NULL,
        started_at TIMESTAMPTZ NOT NULL,
        completed_at TIMESTAMPTZ
    )
    """,
    """
    CREATE TABLE IF NOT EXISTS assessment_retrieval_results
    (
        id UUID PRIMARY KEY,
        retrieval_run_id UUID NOT NULL
            REFERENCES assessment_retrieval_runs(id)
            ON DELETE CASCADE,
        question_id UUID NOT NULL
            REFERENCES assessment_topical_questions(id)
            ON DELETE CASCADE,
        detected_topic TEXT NOT NULL,
        agent1_role VARCHAR(30) NOT NULL,
        requested_official_reference VARCHAR(20) NOT NULL,
        retrieval_stage VARCHAR(50) NOT NULL,
        semantic_score DOUBLE PRECISION NOT NULL,
        final_score DOUBLE PRECISION NOT NULL,
        raw_rank INTEGER,
        unique_rank INTEGER,
        selected BOOLEAN NOT NULL,
        selected_rank INTEGER,
        created_at TIMESTAMPTZ NOT NULL
    )
    """,
    """
    ALTER TABLE assessment_retrieval_runs
    ADD COLUMN IF NOT EXISTS phase2_version VARCHAR(120)
    """,
    """
    ALTER TABLE assessment_retrieval_runs
    ADD COLUMN IF NOT EXISTS phase2_refinement_version VARCHAR(120)
    """,
    """
    ALTER TABLE assessment_retrieval_runs
    ADD COLUMN IF NOT EXISTS phase2_configuration JSONB
    """,
    """
    ALTER TABLE assessment_retrieval_runs
    ADD COLUMN IF NOT EXISTS near_duplicate_count INTEGER
    """,
    """
    ALTER TABLE assessment_retrieval_runs
    ADD COLUMN IF NOT EXISTS phase2_eligible_count INTEGER
    """,
    """
    ALTER TABLE assessment_retrieval_runs
    ADD COLUMN IF NOT EXISTS phase2_rejected_count INTEGER
    """,
    """
    ALTER TABLE assessment_retrieval_runs
    ADD COLUMN IF NOT EXISTS assessment_release_status VARCHAR(50)
    """,
    """
    ALTER TABLE assessment_retrieval_runs
    ADD COLUMN IF NOT EXISTS marks_difference INTEGER
    """,
    """
    ALTER TABLE assessment_retrieval_results
    ADD COLUMN IF NOT EXISTS query_evidence_source VARCHAR(80)
    """,
    """
    ALTER TABLE assessment_retrieval_results
    ADD COLUMN IF NOT EXISTS phase2_semantic_threshold DOUBLE PRECISION
    """,
    """
    ALTER TABLE assessment_retrieval_results
    ADD COLUMN IF NOT EXISTS concept_gate_passed BOOLEAN
    """,
    """
    ALTER TABLE assessment_retrieval_results
    ADD COLUMN IF NOT EXISTS question_quality_gate_passed BOOLEAN
    """,
    """
    ALTER TABLE assessment_retrieval_results
    ADD COLUMN IF NOT EXISTS phase2_gate_passed BOOLEAN
    """,
    """
    ALTER TABLE assessment_retrieval_results
    ADD COLUMN IF NOT EXISTS question_quality_issues JSONB
    """,
    """
    ALTER TABLE assessment_retrieval_results
    ADD COLUMN IF NOT EXISTS phase2_rejection_reasons JSONB
    """,
    """
    ALTER TABLE assessment_retrieval_runs
    ADD COLUMN IF NOT EXISTS phase3_version VARCHAR(120)
    """,
    """
    ALTER TABLE assessment_retrieval_runs
    ADD COLUMN IF NOT EXISTS phase3_review_count INTEGER
    """,
    """
    ALTER TABLE assessment_retrieval_runs
    ADD COLUMN IF NOT EXISTS final_release_blockers JSONB
    """,
    """
    CREATE TABLE IF NOT EXISTS
        assessment_mark_scheme_cleanup_results
    (
        id UUID PRIMARY KEY,
        retrieval_run_id UUID NOT NULL
            REFERENCES assessment_retrieval_runs(id)
            ON DELETE CASCADE,
        question_id UUID NOT NULL
            REFERENCES assessment_topical_questions(id)
            ON DELETE CASCADE,
        mark_scheme_id UUID NOT NULL,
        cleanup_version VARCHAR(120) NOT NULL,
        cleanup_status VARCHAR(50) NOT NULL,
        rule_confidence DOUBLE PRECISION NOT NULL,
        structured_payload JSONB NOT NULL,
        review_reasons JSONB NOT NULL,
        created_at TIMESTAMPTZ NOT NULL
    )
    """,
    """
    CREATE INDEX IF NOT EXISTS
        ix_assessment_ms_cleanup_run
    ON assessment_mark_scheme_cleanup_results
    (
        retrieval_run_id,
        cleanup_status
    )
    """,
    """
    CREATE INDEX IF NOT EXISTS
        ix_assessment_retrieval_results_run
    ON assessment_retrieval_results
    (
        retrieval_run_id,
        selected,
        selected_rank
    )
    """,
]


with engine.begin() as connection:
    for statement in AUDIT_SQL:
        connection.execute(
            text(statement)
        )

print(
    "Retrieval audit tables created/verified."
)


retrieval_run_id = None

if STORE_RETRIEVAL_LOGS:
    audit_metadata = MetaData()

    retrieval_runs = Table(
        "assessment_retrieval_runs",
        audit_metadata,
        autoload_with=engine,
    )

    retrieval_results = Table(
        "assessment_retrieval_results",
        audit_metadata,
        autoload_with=engine,
    )

    mark_scheme_cleanup_results = Table(
        "assessment_mark_scheme_cleanup_results",
        audit_metadata,
        autoload_with=engine,
    )

    retrieval_run_id = uuid.uuid4()

    now_utc = datetime.now(
        timezone.utc
    )

    selected_rank_lookup = {
        str(row["question_id"]): int(
            row["selected_rank"]
        )
        for _, row
        in selected_candidates_df.iterrows()
    }

    selected_ids_set = set(
        selected_rank_lookup
    )

    phase2_configuration = {
        "threshold_strategy": (
            "per_topic_adaptive"
        ),
        "adaptive_threshold_version": (
            ADAPTIVE_THRESHOLD_VERSION
        ),
        "legacy_fixed_strict_threshold": (
            LEGACY_FIXED_STRICT_THRESHOLD
        ),
        "legacy_fixed_relaxed_threshold": (
            LEGACY_FIXED_RELAXED_THRESHOLD
        ),
        "legacy_fixed_thresholds_active": (
            LEGACY_FIXED_THRESHOLD_APPROACH_ENABLED
        ),
        "adaptive_score_percentile": (
            ADAPTIVE_SCORE_PERCENTILE
        ),
        "adaptive_top_score_margin": (
            ADAPTIVE_TOP_SCORE_MARGIN
        ),
        "adaptive_absolute_minimum_score": (
            ADAPTIVE_ABSOLUTE_MINIMUM_SCORE
        ),
        "adaptive_maximum_allowed_threshold": (
            ADAPTIVE_MAXIMUM_ALLOWED_THRESHOLD
        ),
        "adaptive_threshold_min": (
            adaptive_threshold_min
        ),
        "adaptive_threshold_max": (
            adaptive_threshold_max
        ),
        "adaptive_threshold_mean": (
            adaptive_threshold_mean
        ),
        "adaptive_threshold_profile": (
            adaptive_threshold_profile_df
            .to_dict(
                orient="records"
            )
        ),
        "adaptive_pool_sufficient_without_rescue": (
            adaptive_pool_sufficient
        ),
        "near_duplicate_lexical_threshold": (
            NEAR_DUPLICATE_LEXICAL_THRESHOLD
        ),
        "near_duplicate_token_jaccard_threshold": (
            NEAR_DUPLICATE_TOKEN_JACCARD_THRESHOLD
        ),
        "near_duplicate_semantic_threshold": (
            NEAR_DUPLICATE_SEMANTIC_THRESHOLD
        ),
        "topic_coverage_mode": (
            request[
                "topic_coverage_mode"
            ]
        ),
        "required_official_references": (
            request[
                "required_official_references"
            ]
        ),
        "target_marks_tolerance": (
            TARGET_MARKS_TOLERANCE
        ),
    }

    with Session(engine) as session:
        session.execute(
            retrieval_runs.insert().values(
                id=retrieval_run_id,
                retrieval_version=(
                    RETRIEVAL_VERSION
                ),
                specification_code=(
                    SPECIFICATION_CODE
                ),
                specification_version=(
                    SPECIFICATION_VERSION
                ),
                embedding_model=MODEL_NAME,
                qdrant_collection=(
                    AGENT2_COLLECTION
                ),
                agent1_input=json_safe(
                    AGENT1_TOPIC_OUTPUT
                ),
                assessment_request=json_safe(
                    request
                ),
                lesson_summary=(
                    LESSON_SUMMARY
                ),
                raw_candidate_count=len(
                    all_candidates_df
                ),
                duplicate_count=(
                    duplicates_removed
                    + near_duplicates_removed
                ),
                unique_candidate_count=len(
                    near_unique_candidates_df
                ),
                selected_question_count=len(
                    selected_candidates_df
                ),
                selected_total_marks=int(
                    selected_candidates_df[
                        "marks"
                    ].sum()
                ),
                phase2_version=(
                    PHASE2_VERSION
                ),
                phase2_refinement_version=(
                    PHASE2_REFINEMENT_VERSION
                ),
                phase2_configuration=json_safe(
                    phase2_configuration
                ),
                near_duplicate_count=(
                    near_duplicates_removed
                ),
                phase2_eligible_count=len(
                    phase2_candidates_df
                ),
                phase2_rejected_count=len(
                    phase2_rejected_df
                ),
                assessment_release_status=(
                    selection_summary[
                        "assessment_release_status"
                    ]
                ),
                marks_difference=(
                    selection_summary[
                        "marks_difference"
                    ]
                ),
                phase3_version=(
                    PHASE3_VERSION
                ),
                phase3_review_count=(
                    phase3_review_count
                ),
                final_release_blockers=json_safe(
                    selection_summary[
                        "release_blockers"
                    ]
                ),
                status="completed",
                started_at=now_utc,
                completed_at=datetime.now(
                    timezone.utc
                ),
            )
        )

        for _, row in (
            phase2_gate_manifest_df
            .iterrows()
        ):
            question_id = str(
                row["question_id"]
            )

            session.execute(
                retrieval_results
                .insert()
                .values(
                    id=uuid.uuid4(),
                    retrieval_run_id=(
                        retrieval_run_id
                    ),
                    question_id=uuid.UUID(
                        question_id
                    ),
                    detected_topic=(
                        row["detected_topic"]
                    ),
                    agent1_role=(
                        row["agent1_role"]
                    ),
                    requested_official_reference=(
                        row[
                            "requested_official_reference"
                        ]
                    ),
                    retrieval_stage=(
                        row[
                            "retrieval_stage"
                        ]
                    ),
                    semantic_score=float(
                        row["semantic_score"]
                    ),
                    final_score=float(
                        row["final_score"]
                    ),
                    raw_rank=int(
                        row["raw_rank"]
                    ),
                    unique_rank=int(
                        row["unique_rank"]
                    ),
                    selected=(
                        question_id
                        in selected_ids_set
                    ),
                    selected_rank=(
                        selected_rank_lookup.get(
                            question_id
                        )
                    ),
                    query_evidence_source=(
                        row[
                            "query_evidence_source"
                        ]
                    ),
                    phase2_semantic_threshold=(
                        float(
                            row[
                                "phase2_semantic_threshold"
                            ]
                        )
                        if pd.notna(
                            row[
                                "phase2_semantic_threshold"
                            ]
                        )
                        else None
                    ),
                    concept_gate_passed=bool(
                        row[
                            "concept_gate_passed"
                        ]
                    ),
                    question_quality_gate_passed=bool(
                        row[
                            "question_quality_gate_passed"
                        ]
                    ),
                    phase2_gate_passed=bool(
                        row[
                            "phase2_gate_passed"
                        ]
                    ),
                    question_quality_issues=json_safe(
                        row[
                            "question_quality_issues"
                        ]
                    ),
                    phase2_rejection_reasons=json_safe(
                        row[
                            "phase2_rejection_reasons"
                        ]
                    ),
                    created_at=now_utc,
                )
            )

        if (
            PHASE3_STORE_AUDIT_RECORDS
        ):
            for _, row in (
                final_df.iterrows()
            ):
                session.execute(
                    mark_scheme_cleanup_results
                    .insert()
                    .values(
                        id=uuid.uuid4(),
                        retrieval_run_id=(
                            retrieval_run_id
                        ),
                        question_id=uuid.UUID(
                            str(
                                row[
                                    "question_id"
                                ]
                            )
                        ),
                        mark_scheme_id=uuid.UUID(
                            str(
                                row[
                                    "mark_scheme_id"
                                ]
                            )
                        ),
                        cleanup_version=(
                            PHASE3_VERSION
                        ),
                        cleanup_status=(
                            row[
                                "phase3_cleanup_status"
                            ]
                        ),
                        rule_confidence=float(
                            row[
                                "phase3_rule_confidence"
                            ]
                        ),
                        structured_payload=json_safe(
                            {
                                "marking_points": (
                                    row[
                                        "phase3_marking_points"
                                    ]
                                ),
                                "acceptable_answers": (
                                    row[
                                        "phase3_acceptable_answers"
                                    ]
                                ),
                                "rejected_answers": (
                                    row[
                                        "phase3_rejected_answers"
                                    ]
                                ),
                                "additional_guidance": (
                                    row[
                                        "phase3_additional_guidance"
                                    ]
                                ),
                                "worked_examples": (
                                    row[
                                        "phase3_worked_examples"
                                    ]
                                ),
                                "assessment_objectives": (
                                    row[
                                        "phase3_assessment_objectives"
                                    ]
                                ),
                                "block_parser_version": row[
                                    "phase3_block_parser_version"
                                ],
                                "block_count": int(row["phase3_block_count"]),
                                "continuation_lines_merged": int(
                                    row["phase3_continuation_lines_merged"]
                                ),
                                "inline_markers_split": int(
                                    row["phase3_inline_markers_split"]
                                ),
                                "implicit_marking_block_count": int(
                                    row["phase3_implicit_marking_block_count"]
                                ),
                                "ambiguous_lines": row[
                                    "phase3_ambiguous_lines"
                                ],
                                "block_audit": row["phase3_block_audit"],
                            }
                        ),
                        review_reasons=json_safe(
                            row[
                                "phase3_review_reasons"
                            ]
                        ),
                        created_at=now_utc,
                    )
                )

        session.commit()

    print(
        f"Retrieval audit run stored: "
        f"{retrieval_run_id}"
    )

else:
    print(
        "Retrieval logging disabled."
    )


## 16. Final completion checks


In [ ]:
actual_agent1_chunk_evidence_used = bool(
    fallback_evidence_topic_count
    == 0
)

phase3_manifest_created = bool(
    phase3_manifest_path.exists()
)

phase3_json_created = bool(
    phase3_json_path.exists()
)

phase3_raw_guidance_preserved = bool(
    final_df[
        "phase3_raw_guidance_preserved"
    ].all()
)

phase3_block_parser_current = bool(
    final_df["phase3_block_parser_version"].eq(
        PHASE3_BLOCK_PARSER_VERSION
    ).all()
)
phase3_block_metrics_present = bool(
    final_df[
        [
            "phase3_block_count",
            "phase3_continuation_lines_merged",
            "phase3_inline_markers_split",
            "phase3_implicit_marking_block_count",
        ]
    ].notna().all().all()
)
phase3_no_ambiguous_lines = bool(
    final_df["phase3_ambiguous_lines"].map(len).sum() == 0
)
phase3_block_validation_passed = bool(
    all(phase3_block_parser_validation.values())
)

phase3_cleanup_rows_complete = bool(
    len(
        phase3_cleanup_df
    )
    == len(
        final_df
    )
)

phase3_selected_review_count = int(
    (
        final_df[
            "phase3_cleanup_status"
        ]
        == "review_recommended"
    ).sum()
)

selected_candidate_ids = set(
    selected_candidates_df[
        "question_id"
    ].astype(str)
)

selected_near_duplicate_rows = (
    near_duplicate_manifest_df[
        near_duplicate_manifest_df[
            "question_id"
        ].astype(str).isin(
            selected_candidate_ids
        )
    ]
)

selected_near_duplicates_removed = int(
    (
        selected_near_duplicate_rows[
            "near_duplicate_status"
        ]
        == "duplicate_removed"
    ).sum()
)

selected_reference_set = set(
    selected_candidates_df[
        "official_reference"
    ]
    .dropna()
    .astype(str)
    .tolist()
)

required_reference_set = set(
    request.get(
        "required_official_references",
        [],
    )
)

all_required_references_covered = bool(
    required_reference_set.issubset(
        selected_reference_set
    )
)

marks_within_tolerance = bool(
    selection_summary[
        "marks_within_tolerance"
    ]
)

assessment_ready_for_release = bool(
    selection_summary[
        "assessment_release_status"
    ]
    == "ready_for_release"
)

selected_semantic_rescue_count = int(
    selected_candidates_df[
        "semantic_rescue_used"
    ].sum()
)

semantic_rescue_decision_consistent = bool(
    (
        selected_semantic_rescue_count == 0
        and selection_summary[
            "assessment_release_status"
        ]
        in {
            "ready_for_release",
            "needs_user_decision",
        }
    )
    or (
        selected_semantic_rescue_count > 0
        and selection_summary[
            "assessment_release_status"
        ]
        == "needs_user_decision"
    )
)

selected_phase2_gate_passed = bool(
    selected_candidates_df[
        "phase2_gate_passed"
    ].all()
)

selected_concept_or_rescue_passed = bool(
    (
        selected_candidates_df[
            "concept_gate_passed"
        ]
        | selected_candidates_df[
            "semantic_rescue_used"
        ]
    ).all()
)

selected_adaptive_thresholds_present = bool(
    selected_candidates_df[
        "phase2_semantic_threshold"
    ].notna().all()
)

selected_adaptive_thresholds_within_bounds = bool(
    (
        selected_candidates_df[
            "phase2_semantic_threshold"
        ]
        >= ADAPTIVE_ABSOLUTE_MINIMUM_SCORE
    ).all()
    and (
        selected_candidates_df[
            "phase2_semantic_threshold"
        ]
        <= ADAPTIVE_MAXIMUM_ALLOWED_THRESHOLD
    ).all()
)

adaptive_profile_complete = bool(
    len(
        adaptive_threshold_profile_df
    )
    == validated_topics_df[
        "agent1_topic_index"
    ].nunique()
)

legacy_fixed_thresholds_inactive = bool(
    not LEGACY_FIXED_THRESHOLD_APPROACH_ENABLED
)

selected_quality_gate_passed = bool(
    selected_candidates_df[
        "question_quality_gate_passed"
    ].all()
)

selected_quality_issue_count = int(
    selected_candidates_df[
        "question_quality_issues"
    ].map(len).sum()
)

selected_supporting_count = int(
    (
        selected_candidates_df["agent1_role"]
        == "supporting"
    ).sum()
)

selected_distinct_reference_count = int(
    selected_candidates_df[
        "official_reference"
    ].nunique()
)

selected_visual_mask = (
    final_df[
        "has_visual_postgres"
    ].astype(bool)
)

visual_image_lists = (
    final_df.loc[
        selected_visual_mask,
        "rendered_page_images",
    ]
)

all_required_visuals_rendered = bool(
    (
        final_df.loc[
            selected_visual_mask,
            "visual_render_status",
        ]
        == "rendered"
    ).all()
)

all_rendered_image_files_exist = bool(
    all(
        (
            OUTPUT_DIR
            / relative_path
        ).exists()
        for image_list
        in visual_image_lists
        if isinstance(image_list, list)
        for relative_path in image_list
    )
)

selected_texts = (
    final_df[
        "question_text_postgres"
    ].map(normalise_text)
)

checks = {
    "agent1_topics_validated": (
        len(validated_topics_df)
        == len(agent1_topics_df)
    ),
    "all_official_references_known": (
        unknown_topics_df.empty
    ),
    "qdrant_collection_available": (
        AGENT2_COLLECTION
        in collection_names
    ),
    "vector_size_is_384": (
        VECTOR_SIZE
        == EXPECTED_VECTOR_SIZE
    ),
    "exact_candidates_found": (
        len(exact_candidates_df) > 0
    ),
    "enough_phase2_eligible_candidates": (
        len(phase2_candidates_df)
        >= request[
            "number_of_questions"
        ]
    ),
    "requested_question_count_selected": (
        len(selected_candidates_df)
        == request[
            "number_of_questions"
        ]
    ),
    "all_selected_have_mark_schemes": (
        final_df[
            "mark_scheme_id"
        ].notna().all()
    ),
    "no_duplicate_selected_questions": (
        selected_texts.nunique()
        == len(final_df)
    ),
    "candidate_csv_created": (
        candidates_path.exists()
    ),
    "selected_csv_created": (
        selected_path.exists()
    ),
    "json_package_created": (
        package_path.exists()
    ),
    "markdown_package_created": (
        markdown_path.exists()
    ),
    "text_evaluation_report_created": (
        text_report_path.exists()
    ),
    "near_duplicate_manifest_created": (
        near_duplicate_manifest_path.exists()
    ),
    "release_readiness_report_created": (
        release_readiness_path.exists()
    ),
    "no_removed_near_duplicate_selected": (
        selected_near_duplicates_removed
        == 0
    ),
    "all_required_official_references_covered": (
        all_required_references_covered
    ),
    "semantic_rescue_release_decision_consistent": (
        semantic_rescue_decision_consistent
    ),
    "phase2_gate_manifest_created": (
        phase2_gate_manifest_path.exists()
    ),
    "phase2_query_evidence_manifest_created": (
        phase2_query_evidence_path.exists()
    ),
    "all_selected_pass_phase2_gate": (
        selected_phase2_gate_passed
    ),
    "adaptive_threshold_profile_created": (
        adaptive_threshold_profile_path.exists()
    ),
    "adaptive_threshold_profile_complete": (
        adaptive_profile_complete
    ),
    "legacy_fixed_thresholds_inactive": (
        legacy_fixed_thresholds_inactive
    ),
    "selected_adaptive_thresholds_present": (
        selected_adaptive_thresholds_present
    ),
    "selected_adaptive_thresholds_within_bounds": (
        selected_adaptive_thresholds_within_bounds
    ),
    "all_selected_pass_adaptive_gate_or_documented_rescue": (
        selected_concept_or_rescue_passed
    ),
    "all_selected_pass_quality_gate": (
        selected_quality_gate_passed
    ),
    "selected_questions_have_no_quality_issues": (
        selected_quality_issue_count == 0
    ),
    "supporting_requirement_met": (
        selected_supporting_count
        >= request["minimum_supporting_questions"]
    ),
    "distinct_reference_requirement_met": (
        selected_distinct_reference_count
        >= request[
            "minimum_distinct_official_references"
        ]
    ),
    "phase2_full_instruction_segment_scan_active": (
        "probable_truncated_instruction_segment"
        in detect_question_quality_issues(
            (
                "Complete the requested boxes and. "
                "Figure 7"
            )
        )
    ),
    "phase3_manifest_created": (
        phase3_manifest_created
    ),
    "phase3_json_report_created": (
        phase3_json_created
    ),
    "phase3_cleanup_rows_complete": (
        phase3_cleanup_rows_complete
    ),
    "phase3_raw_guidance_preserved": (
        phase3_raw_guidance_preserved
    ),
    "phase3_block_parser_current": (
        phase3_block_parser_current
    ),
    "phase3_block_metrics_present": (
        phase3_block_metrics_present
    ),
    "phase3_block_validation_passed": (
        phase3_block_validation_passed
    ),
    "phase3_no_ambiguous_lines": (
        phase3_no_ambiguous_lines
    ),
    "visual_render_manifest_created": (
        visual_manifest_path.exists()
    ),
    "all_selected_visuals_rendered": (
        all_required_visuals_rendered
    ),
    "all_rendered_image_files_exist": (
        all_rendered_image_files_exist
    ),
    "qdrant_vectors_unchanged_by_phase_1_and_2": (
        qdrant_point_count
        == EXPECTED_QDRANT_POINTS
    ),
}

if STORE_RETRIEVAL_LOGS:
    checks["retrieval_run_logged"] = (
        retrieval_run_id is not None
    )


checks_df = pd.DataFrame(
    [
        {
            "check": name,
            "passed": bool(value),
        }
        for name, value
        in checks.items()
    ]
)

display(checks_df)

notebook_05_complete = all(
    checks.values()
)

release_checks = {
    "marks_within_tolerance": (
        marks_within_tolerance
    ),
    "semantic_rescue_selected": (
        selected_semantic_rescue_count
        > 0
    ),
    "actual_agent1_chunk_evidence_used": (
        actual_agent1_chunk_evidence_used
    ),
    "phase3_review_recommended_count": (
        phase3_selected_review_count
    ),
    "assessment_ready_for_release": (
        assessment_ready_for_release
    ),
}

release_checks_df = pd.DataFrame(
    [
        {
            "release_check": name,
            "passed": bool(value),
        }
        for name, value
        in release_checks.items()
    ]
)

print(
    f"Notebook 05 technically complete: "
    f"{notebook_05_complete}"
)

print(
    "Assessment release status: "
    f"{selection_summary['assessment_release_status']}"
)

display(release_checks_df)

display(
    pd.DataFrame(
        [retrieval_summary]
    )
)


# Notebook 05 — final Phase 1, Phase 2 and Phase 3 criteria

## Technical completion

A successful run should show:

```text
Agent 1 topics validated                                  True
official references known                                True
Qdrant collection available                              True
MiniLM vector size remains 384                            True

near-duplicate manifest created                          True
no removed near duplicate selected                       True

adaptive threshold profile created                       True
legacy fixed 0.60 / 0.55 thresholds inactive            True
selected adaptive thresholds within safety bounds       True

full-question instructional-segment scan active          True
all selected questions pass text-quality gate            True
all required approved references covered                 True

all visual-question pages rendered                       True

Phase 3 manifest created                                 True
Phase 3 JSON report created                              True
one cleanup row per selected mark scheme                 True
raw mark-scheme guidance preserved                       True
Phase 3 block parser version current                     True
wrapped-line validation test passes                      True
block-level audit metrics present                        True
no ambiguous mark-scheme lines remain                    True

JSON/CSV/Markdown/TXT outputs created                    True
retrieval and cleanup audit records stored               True
Qdrant point count remains 820                           True

Notebook 05 technically complete                         True
```

## Release-state interpretation

```text
ready_for_release
    marks are within tolerance
    no semantic rescue was selected
    actual Agent 1 source chunk text was used
    Phase 3 does not recommend review

evaluation_ready
    technical retrieval is complete
    lesson-summary fallback was used or Phase 3 recommends review

needs_user_decision
    marks are outside tolerance or semantic rescue was selected
```

The current standalone sample normally remains `evaluation_ready` because it does
not yet receive actual source chunk text from Agent 1 Streamlit.

## Phase 3 interpretation

```text
raw marking_guidance             source of truth
legacy structured fields         retained for comparison
Phase 3 structured fields        cleaner interface view
review_recommended               requires human checking
structured_ready                 suitable for evaluation display
```

## Remaining work

```text
Actual Agent 1 source-chunk integration test             Streamlit integration
Adaptive-parameter comparison                            Notebook 06
Human retrieval relevance ratings                        Notebook 06
Phase 3 rule evaluation across more mark schemes         Notebook 06 / review
```

## Final Phase 3 refinement status

```text
Line-by-line classification approach                    tested
Wrapped-line classification issue                       documented
Inline A. / I. / R. marker splitting                    implemented
Logical block construction                              implemented
Guidance continuation merging                           implemented
Marking-point continuation merging                      implemented
Worked-example boundaries                               preserved
Synthetic wrapped-line validation                       added
Raw guidance preservation                               retained
```

After this notebook is verified, the next development step is Agent 1 Streamlit and
Agent 2 retrieval integration.
